# 05 — Temporally Aligned Hierarchical Hail Modeling

## Purpose

Notebook 04 showed that retrospective storm conditioning does not provide a robust ERA5-only advantage for distinguishing hail-producing storms from non-hail storms.

This notebook changes the question.

Instead of predicting hail within the same retrospective analysis hour, the experiment defines a forecast origin $t_0$ and imposes explicit temporal ordering:

$$
X^-
\rightarrow
S^+
\rightarrow
H^+.
$$

Here,

$$
X^-
$$

contains predictors whose **valid times do not extend beyond the forecast origin**. Pre-$t_0$ MRMS predictors are constructed only from radar scans before $t_0$. ERA5 predictors are retrospective reanalysis fields, so this temporal ordering should be interpreted as **valid-time alignment rather than operational real-time availability**.

$$
S^+
$$

is radar-defined storm occurrence during the future target window, and

$$
H^+
$$

is observed NOAA hail occurrence during the same future target window.

The direct formulation is

$$
P(H^+=1 \mid X^-).
$$

The hierarchical formulation is

$$
P(S^+=1 \mid X^-)
\,
P(H^+=1 \mid S^+=1, X^-).
$$

The objective is not to force the hierarchical formulation to outperform the direct model.

The objective is to determine whether predictive information enters primarily through future storm occurrence, through hail production within future storms, or through both stages.

## 1. Temporally ordered pilot design

Two forecast-origin regimes are evaluated.

### 30/30 regime

The forecast origin is 30 minutes after the beginning of each radar period.

Radar predictors use only the preceding 30 minutes:

$$
[t_0-30\text{ min},\,t_0).
$$

Future storm and hail outcomes use the following 30 minutes:

$$
(t_0,\,t_0+30\text{ min}].
$$

### 45/30 regime

The forecast origin is 45 minutes after the beginning of each radar period.

The same 30-minute pre-origin lookback is used:

$$
[t_0-30\text{ min},\,t_0),
$$

while the future target window remains

$$
(t_0,\,t_0+30\text{ min}].
$$

The 45/30 regime therefore represents a later storm state while preserving the same forecast horizon.

### Evaluation

The current experiment contains nine event-enriched 2024 periods.

Entire periods are held out using leave-one-period-out validation.

The same held-out rows and predictor set are used for the direct and hierarchical formulations.

The experiment enforces temporal ordering between predictors and future outcomes, but it is retrospective: ERA5 variables are reanalysis fields aligned by valid time rather than operationally available forecast products.

Because the nine periods were selected for methodological development rather than by a representative climatological sampling rule, this notebook is treated as a **temporally ordered retrospective development pilot**, not as a population-level probability evaluation or an operational forecasting validation.

## 2. Implementation corrections inherited from the exploratory notebook

The original exploratory future-window implementation preceded the final Notebook 03 storm-rule audit and contained two implementation issues that are corrected before any results in this notebook are interpreted.

### 2.1 Fine-pixel support denominator

The exploratory implementation computed coarse-cell reflectivity support using the fraction of **available** fine pixels:

```text
coarsen(...).mean(skipna=True)
```

This can inflate spatial support when only part of a nominal 0.25° cell contains valid radar data.

Notebook 03 subsequently froze the corrected storm definition:

1. MRMS missing sentinels `-99` and `-999` are both treated as unavailable;
2. a scan-cell is usable only when at least 80% of its nominal fine pixels are valid;
3. 35-dBZ exceedance support is divided by the **full nominal 625-pixel coarse cell**;
4. reflectivity support is masked for scan-cells that fail the fine-pixel coverage requirement;
5. a radar window is eligible only when at least 80% of its scans satisfy the per-scan coverage criterion.

The temporally aligned experiment therefore recomputes all radar summaries and future-storm labels from the local MRMS inputs using this corrected definition.

### 2.2 Cross-midnight future-window completeness

The exploratory MRMS file-query helper did not reliably retrieve scans from every UTC calendar date touched by a requested interval.

This matters for the 45/30 regime because a period beginning late in the UTC day can have a future target window that extends past midnight. For example, the `expand_active_20240613_23` 45/30 target window is

$$
(2024\text{-}06\text{-}13\ 23{:}45,\,
2024\text{-}06\text{-}14\ 00{:}15].
$$

The corrected implementation queries every UTC date touched by the requested interval and independently audits temporal completeness before constructing radar summaries.

For every period-origin pair used below, the notebook verifies:

- the expected 168 canonical grid cells;
- 15 scans in the 30-minute pre-origin window;
- 15 scans in the 30-minute future target window;
- no duplicate period-origin-grid keys.

Historical exploratory 30/30 and 45/30 metric values are therefore treated only as exploratory context. All reported results in this notebook are recomputed after both implementation corrections.

In [1]:
from pathlib import Path
import gzip
import re
import shutil

import numpy as np
import pandas as pd
import xarray as xr

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)


# ------------------------------------------------------------
# Locate repository root
# ------------------------------------------------------------

cwd = Path.cwd()


if (
    cwd
    / "data"
).exists():

    REPO_ROOT = cwd


elif (
    cwd.parent
    / "data"
).exists():

    REPO_ROOT = cwd.parent


else:

    raise FileNotFoundError(
        "Could not locate repository root. "
        "Run this notebook from the repository root "
        "or notebooks/."
    )


DATA_DIR = (
    REPO_ROOT
    / "data"
)


MRMS_DIR = (
    DATA_DIR
    / "mrms"
)


ERA5_DIR = (
    DATA_DIR
    / "era5"
)


HAIL_PATH = (
    DATA_DIR
    / "hail_records"
    / "Combined US Hail Data.csv"
)


OUTPUT_TABLE_DIR = (
    REPO_ROOT
    / "outputs"
    / "tables"
)


OUTPUT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Repository interface initialized."
)

Repository interface initialized.


In [2]:
# ------------------------------------------------------------
# Spatial domain
# ------------------------------------------------------------

LAT_MIN = 37.5
LAT_MAX = 40.5

LON_MIN = -92.0
LON_MAX = -88.5


# ------------------------------------------------------------
# MRMS grid and storm definition
#
# Frozen from Notebook 03
# ------------------------------------------------------------

GRID_FACTOR = 25

COARSE_GRID_DEG = 0.25

NOMINAL_PIXELS_PER_CELL = (
    GRID_FACTOR
    *
    GRID_FACTOR
)


FINE_PIXEL_COVERAGE_THRESHOLD = 0.80

GRID_HOUR_COVERAGE_THRESHOLD = 0.80

REFLECTIVITY_THRESHOLD_DBZ = 35.0

STRICT_AREA_FRACTION = 0.10

STRICT_MIN_SCANS = 5


# ------------------------------------------------------------
# Temporal design
# ------------------------------------------------------------

FORECAST_ORIGIN_MINUTES = [
    30,
    45,
]

LOOKBACK_MINUTES = 30

TARGET_MINUTES = 30


# ------------------------------------------------------------
# Final compact predictor specification
# ------------------------------------------------------------

ERA5_FEATURES = [
    "cape",
    "t2m",
    "d2m",
    "t_500",
    "shear_850_500",
    "shear_850_300",
]


PRE_T0_RADAR_FEATURES = [
    "pre_cmax_dbz",
    "pre_n_area5",
    "pre_n_area10",
]


FINAL_FEATURES = (
    ERA5_FEATURES
    +
    PRE_T0_RADAR_FEATURES
)


print(
    "Nominal fine pixels per coarse cell:",
    NOMINAL_PIXELS_PER_CELL,
)

print(
    "Forecast origins:",
    FORECAST_ORIGIN_MINUTES,
)

print(
    "Final predictor count:",
    len(
        FINAL_FEATURES
    ),
)

Nominal fine pixels per coarse cell: 625
Forecast origins: [30, 45]
Final predictor count: 9


## 3. Nine-period development panel

The temporally aligned pilot uses nine 2024 periods.

Three periods come from the original Notebook 03 validation panel:

- `active_20240418`
- `active_20240508`
- `active_20240526`

Six additional hail-active periods were identified during the exploratory expansion:

- `expand_active_20240210_07`
- `expand_active_20240502_21`
- `expand_active_20240520_01`
- `expand_active_20240523_10`
- `expand_active_20240609_07`
- `expand_active_20240613_23`

The expanded periods increase event diversity but do not turn the sample into a representative climatological panel.

All nine periods are therefore treated as development / directional evidence.

No period is split across training and testing folds.

In [3]:
# ------------------------------------------------------------
# Frozen nine-period registry
# ------------------------------------------------------------

PERIOD_PLAN = pd.DataFrame(
    {
        "period_id": [
            "active_20240418",
            "active_20240508",
            "active_20240526",

            "expand_active_20240210_07",
            "expand_active_20240502_21",
            "expand_active_20240520_01",
            "expand_active_20240523_10",
            "expand_active_20240609_07",
            "expand_active_20240613_23",
        ],

        "start_utc": pd.to_datetime(
            [
                "2024-04-18 22:00:00",
                "2024-05-08 21:00:00",
                "2024-05-26 22:00:00",

                "2024-02-10 07:00:00",
                "2024-05-02 21:00:00",
                "2024-05-20 01:00:00",
                "2024-05-23 10:00:00",
                "2024-06-09 07:00:00",
                "2024-06-13 23:00:00",
            ]
        ),

        "panel_group": [
            "primary",
            "primary",
            "primary",

            "expanded",
            "expanded",
            "expanded",
            "expanded",
            "expanded",
            "expanded",
        ],
    }
)


# ------------------------------------------------------------
# Local MRMS locations
# ------------------------------------------------------------

PERIOD_PLAN[
    "radar_dir"
] = PERIOD_PLAN.apply(
    lambda row:
        (
            MRMS_DIR
            / (
                "validation"
                if row[
                    "panel_group"
                ]
                == "primary"
                else
                "expanded_active"
            )
            / row[
                "period_id"
            ]
        ),
    axis=1,
)


# ------------------------------------------------------------
# Local ERA5 locations
#
# Primary periods reuse the ERA5 files already used by
# Notebook 04.
#
# Expanded periods reuse the previously downloaded
# quick_expanded files.
# ------------------------------------------------------------

PERIOD_PLAN[
    "era5_dir"
] = PERIOD_PLAN[
    "panel_group"
].map(
    {
        "primary":
            ERA5_DIR
            / "pilot_2024",

        "expanded":
            ERA5_DIR
            / "quick_expanded",
    }
)


PERIOD_PLAN[
    "era5_single_file"
] = PERIOD_PLAN.apply(
    lambda row:
        row[
            "era5_dir"
        ]
        / (
            row[
                "period_id"
            ]
            + "_single.nc"
        ),
    axis=1,
)


PERIOD_PLAN[
    "era5_pressure_file"
] = PERIOD_PLAN.apply(
    lambda row:
        row[
            "era5_dir"
        ]
        / (
            row[
                "period_id"
            ]
            + "_pressure.nc"
        ),
    axis=1,
)


# ------------------------------------------------------------
# Public-facing registry
# ------------------------------------------------------------

display(
    PERIOD_PLAN[
        [
            "period_id",
            "start_utc",
            "panel_group",
        ]
    ]
)

,period_id,start_utc,panel_group
0,active_20240418,2024-04-18 22:00:00,primary
1,active_20240508,2024-05-08 21:00:00,primary
2,active_20240526,2024-05-26 22:00:00,primary
3,expand_active_20240210_07,2024-02-10 07:00:00,expanded
4,expand_active_20240502_21,2024-05-02 21:00:00,expanded
5,expand_active_20240520_01,2024-05-20 01:00:00,expanded
6,expand_active_20240523_10,2024-05-23 10:00:00,expanded
7,expand_active_20240609_07,2024-06-09 07:00:00,expanded
8,expand_active_20240613_23,2024-06-13 23:00:00,expanded


In [4]:
# ------------------------------------------------------------
# Local input availability audit
# ------------------------------------------------------------

input_audit_rows = []


for _, row in (
    PERIOD_PLAN
    .iterrows()
):

    radar_dir = (
        row[
            "radar_dir"
        ]
    )

    single_file = (
        row[
            "era5_single_file"
        ]
    )

    pressure_file = (
        row[
            "era5_pressure_file"
        ]
    )


    radar_gz_count = (
        len(
            list(
                radar_dir.glob(
                    "*.grib2.gz"
                )
            )
        )
        if radar_dir.exists()
        else 0
    )


    input_audit_rows.append(
        {
            "period_id":
                row[
                    "period_id"
                ],

            "panel_group":
                row[
                    "panel_group"
                ],

            "radar_directory_exists":
                radar_dir.exists(),

            "radar_gz_files":
                radar_gz_count,

            "era5_single_exists":
                single_file.exists(),

            "era5_pressure_exists":
                pressure_file.exists(),
        }
    )


input_audit = pd.DataFrame(
    input_audit_rows
)


display(
    input_audit
)


print(
    "\nNOAA hail source exists:",
    HAIL_PATH.exists(),
)

,period_id,panel_group,radar_directory_exists,radar_gz_files,era5_single_exists,era5_pressure_exists
0,active_20240418,primary,True,38,True,True
1,active_20240508,primary,True,38,True,True
2,active_20240526,primary,True,38,True,True
3,expand_active_20240210_07,expanded,True,38,True,True
4,expand_active_20240502_21,expanded,True,38,True,True
5,expand_active_20240520_01,expanded,True,38,True,True
6,expand_active_20240523_10,expanded,True,38,True,True
7,expand_active_20240609_07,expanded,True,38,True,True
8,expand_active_20240613_23,expanded,True,38,True,True



NOAA hail source exists: True


In [5]:
# ------------------------------------------------------------
# Cross-date MRMS file-table helper
#
# Unlike the exploratory helper, this version queries every
# UTC calendar date touched by [start_utc, end_utc].
# ------------------------------------------------------------

import requests
import xml.etree.ElementTree as ET


MRMS_BUCKET_URL = (
    "https://noaa-mrms-pds.s3.amazonaws.com"
)


MRMS_PRODUCT_PREFIX = (
    "CONUS/"
    "MergedReflectivityQCComposite_00.50/"
)


def get_mrms_file_table(
    period_id,
    start_utc,
    end_utc,
):
    """
    Return MRMS composite-reflectivity files whose timestamps
    fall inside [start_utc, end_utc].

    All UTC calendar dates touched by the requested interval
    are queried separately, so windows crossing midnight are
    handled correctly.
    """

    start_utc = pd.Timestamp(
        start_utc
    )

    end_utc = pd.Timestamp(
        end_utc
    )


    if end_utc < start_utc:

        raise ValueError(
            "end_utc must be greater than or equal to start_utc."
        )


    query_dates = pd.date_range(
        start=
            start_utc.normalize(),

        end=
            end_utc.normalize(),

        freq="D",
    )


    rows = []


    for query_date in (
        query_dates
    ):

        date_str = (
            query_date
            .strftime(
                "%Y%m%d"
            )
        )


        prefix = (
            MRMS_PRODUCT_PREFIX
            + date_str
            + "/"
        )


        continuation_token = None


        while True:

            params = {
                "list-type":
                    "2",

                "prefix":
                    prefix,
            }


            if (
                continuation_token
                is not None
            ):

                params[
                    "continuation-token"
                ] = continuation_token


            response = requests.get(
                MRMS_BUCKET_URL,
                params=params,
                timeout=60,
            )


            response.raise_for_status()


            root = ET.fromstring(
                response.content
            )


            namespace = {
                "s3":
                    "http://s3.amazonaws.com/doc/2006-03-01/"
            }


            for contents in (
                root.findall(
                    "s3:Contents",
                    namespace,
                )
            ):

                key_node = contents.find(
                    "s3:Key",
                    namespace,
                )


                if key_node is None:

                    continue


                key = key_node.text

                filename = (
                    key
                    .split("/")[-1]
                )


                timestamp_match = re.search(
                    r"(\d{8}-\d{6})",
                    filename,
                )


                if timestamp_match is None:

                    continue


                radar_time = pd.to_datetime(
                    timestamp_match.group(1),
                    format="%Y%m%d-%H%M%S",
                    errors="coerce",
                )


                if pd.isna(
                    radar_time
                ):

                    continue


                if (
                    radar_time
                    >= start_utc
                    and
                    radar_time
                    <= end_utc
                ):

                    rows.append(
                        {
                            "period_id":
                                period_id,

                            "filename":
                                filename,

                            "radar_time_utc":
                                radar_time,

                            "key":
                                key,

                            "source_date":
                                date_str,
                        }
                    )


            is_truncated_node = root.find(
                "s3:IsTruncated",
                namespace,
            )


            is_truncated = (
                is_truncated_node
                is not None
                and
                is_truncated_node.text
                == "true"
            )


            if not is_truncated:

                break


            token_node = root.find(
                "s3:NextContinuationToken",
                namespace,
            )


            if (
                token_node is None
                or
                token_node.text is None
            ):

                raise RuntimeError(
                    "S3 listing reported truncation "
                    "without a continuation token."
                )


            continuation_token = (
                token_node.text
            )


    file_table = pd.DataFrame(
        rows,
        columns=[
            "period_id",
            "filename",
            "radar_time_utc",
            "key",
            "source_date",
        ],
    )


    if len(
        file_table
    ) > 0:

        file_table = (
            file_table
            .drop_duplicates(
                subset=[
                    "key",
                ]
            )
            .sort_values(
                "radar_time_utc"
            )
            .reset_index(
                drop=True
            )
        )


    return file_table


# ------------------------------------------------------------
# Cross-midnight sanity check only
#
# No files are downloaded here.
# ------------------------------------------------------------

june13_test = (
    get_mrms_file_table(
        period_id=
            "expand_active_20240613_23",

        start_utc=
            pd.Timestamp(
                "2024-06-13 23:00:00"
            ),

        end_utc=
            pd.Timestamp(
                "2024-06-14 00:15:00"
            ),
    )
)


print(
    "Scans found:",
    len(
        june13_test
    ),
)


print(
    "First scan:",
    (
        june13_test[
            "radar_time_utc"
        ].min()
        if len(
            june13_test
        ) > 0
        else pd.NaT
    ),
)


print(
    "Last scan:",
    (
        june13_test[
            "radar_time_utc"
        ].max()
        if len(
            june13_test
        ) > 0
        else pd.NaT
    ),
)


print(
    "UTC dates represented:",
    (
        june13_test[
            "source_date"
        ]
        .drop_duplicates()
        .tolist()
    ),
)

Scans found: 38
First scan: 2024-06-13 23:00:40
Last scan: 2024-06-14 00:14:39
UTC dates represented: ['20240613', '20240614']


In [6]:
# ------------------------------------------------------------
# Temporal-window completeness audit
#
# This checks the local MRMS scan timestamps before any radar
# decoding or label construction.
#
# A 30-minute window is considered temporally complete when:
#   1. at least 12 scans are present;
#   2. the first scan is within 3 minutes of the window start;
#   3. the last scan is within 3 minutes of the window end;
#   4. no internal scan gap exceeds 4 minutes.
#
# These checks are separate from the later spatial / valid-pixel
# MRMS coverage criterion.
# ------------------------------------------------------------

TEMPORAL_EDGE_TOLERANCE = pd.Timedelta(
    minutes=3
)

MAX_SCAN_GAP = pd.Timedelta(
    minutes=4
)

MIN_SCANS_PER_30MIN_WINDOW = 12


def local_scan_times(
    radar_dir
):
    """
    Parse MRMS UTC timestamps from local .grib2.gz filenames.
    """

    scan_times = []


    for path in sorted(
        radar_dir.glob(
            "*.grib2.gz"
        )
    ):

        timestamp_match = re.search(
            r"(\d{8}-\d{6})",
            path.name,
        )


        if timestamp_match is None:

            continue


        scan_time = pd.to_datetime(
            timestamp_match.group(1),
            format="%Y%m%d-%H%M%S",
            errors="coerce",
        )


        if pd.notna(
            scan_time
        ):

            scan_times.append(
                scan_time
            )


    return pd.DatetimeIndex(
        scan_times
    ).sort_values()


def audit_30min_window(
    scan_times,
    window_start,
    window_end,
    left_closed,
):
    """
    Audit one 30-minute MRMS window.

    left_closed=True:
        [window_start, window_end)

    left_closed=False:
        (window_start, window_end]
    """

    if left_closed:

        window_times = scan_times[
            (
                scan_times
                >= window_start
            )
            &
            (
                scan_times
                < window_end
            )
        ]

    else:

        window_times = scan_times[
            (
                scan_times
                > window_start
            )
            &
            (
                scan_times
                <= window_end
            )
        ]


    n_scans = len(
        window_times
    )


    if n_scans == 0:

        return {
            "n_scans":
                0,

            "first_scan":
                pd.NaT,

            "last_scan":
                pd.NaT,

            "max_gap_minutes":
                np.nan,

            "start_edge_ok":
                False,

            "end_edge_ok":
                False,

            "scan_count_ok":
                False,

            "max_gap_ok":
                False,

            "window_complete":
                False,
        }


    first_scan = (
        window_times.min()
    )

    last_scan = (
        window_times.max()
    )


    if n_scans >= 2:

        scan_gaps = (
            pd.Series(
                window_times
            )
            .diff()
            .dropna()
        )

        max_gap = (
            scan_gaps.max()
        )

    else:

        max_gap = pd.NaT


    start_edge_ok = (
        first_scan
        <=
        window_start
        +
        TEMPORAL_EDGE_TOLERANCE
    )


    end_edge_ok = (
        last_scan
        >=
        window_end
        -
        TEMPORAL_EDGE_TOLERANCE
    )


    scan_count_ok = (
        n_scans
        >=
        MIN_SCANS_PER_30MIN_WINDOW
    )


    max_gap_ok = (
        pd.notna(
            max_gap
        )
        and
        max_gap
        <=
        MAX_SCAN_GAP
    )


    window_complete = (
        start_edge_ok
        and
        end_edge_ok
        and
        scan_count_ok
        and
        max_gap_ok
    )


    return {
        "n_scans":
            n_scans,

        "first_scan":
            first_scan,

        "last_scan":
            last_scan,

        "max_gap_minutes":
            (
                max_gap.total_seconds()
                / 60
                if pd.notna(
                    max_gap
                )
                else np.nan
            ),

        "start_edge_ok":
            start_edge_ok,

        "end_edge_ok":
            end_edge_ok,

        "scan_count_ok":
            scan_count_ok,

        "max_gap_ok":
            max_gap_ok,

        "window_complete":
            window_complete,
    }


temporal_audit_rows = []


for _, row in (
    PERIOD_PLAN
    .iterrows()
):

    period_id = (
        row[
            "period_id"
        ]
    )

    start_utc = pd.Timestamp(
        row[
            "start_utc"
        ]
    )

    scan_times = (
        local_scan_times(
            row[
                "radar_dir"
            ]
        )
    )


    for origin_minutes in (
        FORECAST_ORIGIN_MINUTES
    ):

        t0 = (
            start_utc
            +
            pd.Timedelta(
                minutes=
                    origin_minutes
            )
        )

        pre_start = (
            t0
            -
            pd.Timedelta(
                minutes=
                    LOOKBACK_MINUTES
            )
        )

        target_end = (
            t0
            +
            pd.Timedelta(
                minutes=
                    TARGET_MINUTES
            )
        )


        pre_audit = (
            audit_30min_window(
                scan_times=
                    scan_times,

                window_start=
                    pre_start,

                window_end=
                    t0,

                left_closed=
                    True,
            )
        )


        future_audit = (
            audit_30min_window(
                scan_times=
                    scan_times,

                window_start=
                    t0,

                window_end=
                    target_end,

                left_closed=
                    False,
            )
        )


        temporal_audit_rows.append(
            {
                "period_id":
                    period_id,

                "origin_minutes":
                    origin_minutes,

                "pre_n_scans":
                    pre_audit[
                        "n_scans"
                    ],

                "pre_first_scan":
                    pre_audit[
                        "first_scan"
                    ],

                "pre_last_scan":
                    pre_audit[
                        "last_scan"
                    ],

                "pre_max_gap_min":
                    pre_audit[
                        "max_gap_minutes"
                    ],

                "pre_complete":
                    pre_audit[
                        "window_complete"
                    ],

                "future_n_scans":
                    future_audit[
                        "n_scans"
                    ],

                "future_first_scan":
                    future_audit[
                        "first_scan"
                    ],

                "future_last_scan":
                    future_audit[
                        "last_scan"
                    ],

                "future_max_gap_min":
                    future_audit[
                        "max_gap_minutes"
                    ],

                "future_complete":
                    future_audit[
                        "window_complete"
                    ],
            }
        )


temporal_audit = pd.DataFrame(
    temporal_audit_rows
)


display(
    temporal_audit
)


print(
    "\nAll pre-origin windows complete:",
    temporal_audit[
        "pre_complete"
    ].all(),
)


print(
    "All future windows complete:",
    temporal_audit[
        "future_complete"
    ].all(),
)

,period_id,origin_minutes,pre_n_scans,pre_first_scan,pre_last_scan,pre_max_gap_min,pre_complete,future_n_scans,future_first_scan,future_last_scan,future_max_gap_min,future_complete
0,active_20240418,30,15,2024-04-18 22:00:39,2024-04-18 22:28:42,2.066667,True,15,2024-04-18 22:30:40,2024-04-18 22:58:40,2.083333,True
1,active_20240418,45,15,2024-04-18 22:16:39,2024-04-18 22:44:43,2.083333,True,15,2024-04-18 22:46:40,2024-04-18 23:14:40,2.050000,True
2,active_20240508,30,15,2024-05-08 21:00:42,2024-05-08 21:28:42,2.050000,True,15,2024-05-08 21:30:40,2024-05-08 21:58:38,2.100000,True
3,active_20240508,45,15,2024-05-08 21:16:43,2024-05-08 21:44:42,2.100000,True,15,2024-05-08 21:46:41,2024-05-08 22:14:41,2.083333,True
4,active_20240526,30,15,2024-05-26 22:00:41,2024-05-26 22:28:39,2.066667,True,15,2024-05-26 22:30:42,2024-05-26 22:58:39,2.033333,True
5,active_20240526,45,15,2024-05-26 22:16:40,2024-05-26 22:44:40,2.066667,True,15,2024-05-26 22:46:39,2024-05-26 23:14:39,2.066667,True
6,expand_active_20240210_07,30,15,2024-02-10 07:00:39,2024-02-10 07:28:39,2.100000,True,15,2024-02-10 07:30:38,2024-02-10 07:58:40,2.133333,True
7,expand_active_20240210_07,45,15,2024-02-10 07:16:37,2024-02-10 07:44:38,2.100000,True,15,2024-02-10 07:46:39,2024-02-10 08:14:39,2.133333,True
8,expand_active_20240502_21,30,15,2024-05-02 21:00:42,2024-05-02 21:28:38,2.033333,True,15,2024-05-02 21:30:42,2024-05-02 21:58:40,2.066667,True
9,expand_active_20240502_21,45,15,2024-05-02 21:16:39,2024-05-02 21:44:41,2.066667,True,15,2024-05-02 21:46:42,2024-05-02 22:14:38,2.066667,True



All pre-origin windows complete: True
All future windows complete: True


In [7]:
# ------------------------------------------------------------
# ERA5 schema audit
#
# Inspect one primary-period pair and one expanded-period pair
# before writing the common ERA5 feature extractor.
# ------------------------------------------------------------

schema_examples = (
    PERIOD_PLAN
    .groupby(
        "panel_group",
        sort=False,
    )
    .head(1)
    .reset_index(drop=True)
)


for _, row in (
    schema_examples
    .iterrows()
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        "PERIOD:",
        row[
            "period_id"
        ],
    )

    print(
        "GROUP:",
        row[
            "panel_group"
        ],
    )


    single_path = (
        row[
            "era5_single_file"
        ]
    )

    pressure_path = (
        row[
            "era5_pressure_file"
        ]
    )


    with xr.open_dataset(
        single_path
    ) as ds_single:

        print(
            "\nSINGLE-LEVEL FILE"
        )

        print(
            "Path:",
            single_path
        )

        print(
            "Dimensions:",
            dict(
                ds_single.sizes
            )
        )

        print(
            "Coordinates:",
            list(
                ds_single.coords
            )
        )

        print(
            "Data variables:",
            list(
                ds_single.data_vars
            )
        )


        time_coord = (
            "valid_time"
            if "valid_time" in ds_single.coords
            else
            "time"
            if "time" in ds_single.coords
            else
            None
        )


        if time_coord is not None:

            print(
                "Time values:",
                pd.to_datetime(
                    ds_single[
                        time_coord
                    ].values
                ),
            )


    with xr.open_dataset(
        pressure_path
    ) as ds_pressure:

        print(
            "\nPRESSURE-LEVEL FILE"
        )

        print(
            "Path:",
            pressure_path
        )

        print(
            "Dimensions:",
            dict(
                ds_pressure.sizes
            )
        )

        print(
            "Coordinates:",
            list(
                ds_pressure.coords
            )
        )

        print(
            "Data variables:",
            list(
                ds_pressure.data_vars
            )
        )


        level_coord = (
            "pressure_level"
            if "pressure_level"
            in ds_pressure.coords
            else
            "level"
            if "level"
            in ds_pressure.coords
            else
            None
        )


        if level_coord is not None:

            print(
                "Pressure levels:",
                ds_pressure[
                    level_coord
                ].values,
            )


        time_coord = (
            "valid_time"
            if "valid_time"
            in ds_pressure.coords
            else
            "time"
            if "time"
            in ds_pressure.coords
            else
            None
        )


        if time_coord is not None:

            print(
                "Time values:",
                pd.to_datetime(
                    ds_pressure[
                        time_coord
                    ].values
                ),
            )


PERIOD: active_20240418
GROUP: primary

SINGLE-LEVEL FILE
Path: data/era5/pilot_2024/active_20240418_single.nc
Dimensions: {'valid_time': 1, 'latitude': 13, 'longitude': 15}
Coordinates: ['number', 'valid_time', 'latitude', 'longitude', 'expver']
Data variables: ['t2m', 'd2m', 'cape', 'cin']
Time values: DatetimeIndex(['2024-04-18 22:00:00'], dtype='datetime64[ns]', freq=None)

PRESSURE-LEVEL FILE
Path: data/era5/pilot_2024/active_20240418_pressure.nc
Dimensions: {'valid_time': 1, 'pressure_level': 4, 'latitude': 13, 'longitude': 15}
Coordinates: ['number', 'valid_time', 'pressure_level', 'latitude', 'longitude', 'expver']
Data variables: ['t', 'q', 'u', 'v']
Pressure levels: [850. 700. 500. 300.]
Time values: DatetimeIndex(['2024-04-18 22:00:00'], dtype='datetime64[ns]', freq=None)

PERIOD: expand_active_20240210_07
GROUP: expanded

SINGLE-LEVEL FILE
Path: data/era5/quick_expanded/expand_active_20240210_07_single.nc
Dimensions: {'valid_time': 1, 'latitude': 13, 'longitude': 15}
Coord

In [8]:
# ------------------------------------------------------------
# Compact expanded-ERA5 schema consistency audit
# ------------------------------------------------------------

expanded_schema_rows = []


for _, row in (
    PERIOD_PLAN[
        PERIOD_PLAN[
            "panel_group"
        ]
        == "expanded"
    ]
    .iterrows()
):

    period_id = (
        row[
            "period_id"
        ]
    )

    start_utc = pd.Timestamp(
        row[
            "start_utc"
        ]
    )


    with xr.open_dataset(
        row[
            "era5_single_file"
        ]
    ) as ds_single:

        single_vars = sorted(
            list(
                ds_single.data_vars
            )
        )

        single_time = pd.to_datetime(
            ds_single[
                "valid_time"
            ].values
        )[0]


    with xr.open_dataset(
        row[
            "era5_pressure_file"
        ]
    ) as ds_pressure:

        pressure_vars = sorted(
            list(
                ds_pressure.data_vars
            )
        )

        pressure_levels = (
            ds_pressure[
                "pressure_level"
            ]
            .values
            .tolist()
        )

        pressure_time = pd.to_datetime(
            ds_pressure[
                "valid_time"
            ].values
        )[0]


    expanded_schema_rows.append(
        {
            "period_id":
                period_id,

            "single_vars":
                single_vars,

            "pressure_vars":
                pressure_vars,

            "pressure_levels":
                pressure_levels,

            "single_time":
                single_time,

            "pressure_time":
                pressure_time,

            "single_time_matches_start":
                single_time
                == start_utc,

            "pressure_time_matches_start":
                pressure_time
                == start_utc,

            "required_single_present":
                {
                    "t2m",
                    "d2m",
                    "cape",
                }
                .issubset(
                    single_vars
                ),

            "required_pressure_present":
                {
                    "t",
                    "u",
                    "v",
                }
                .issubset(
                    pressure_vars
                ),

            "required_levels_present":
                {
                    850.0,
                    500.0,
                    300.0,
                }
                .issubset(
                    pressure_levels
                ),
        }
    )


expanded_schema_audit = pd.DataFrame(
    expanded_schema_rows
)


display(
    expanded_schema_audit
)


print(
    "\nAll expanded ERA5 inputs satisfy required schema:",
    (
        expanded_schema_audit[
            [
                "single_time_matches_start",
                "pressure_time_matches_start",
                "required_single_present",
                "required_pressure_present",
                "required_levels_present",
            ]
        ]
        .all()
        .all()
    ),
)

,period_id,single_vars,pressure_vars,pressure_levels,single_time,pressure_time,single_time_matches_start,pressure_time_matches_start,required_single_present,required_pressure_present,required_levels_present
0,expand_active_20240210_07,"[cape, d2m, t2m]","[t, u, v]","[850.0, 500.0, 300.0]",2024-02-10 07:00:00,2024-02-10 07:00:00,True,True,True,True,True
1,expand_active_20240502_21,"[cape, d2m, t2m]","[t, u, v]","[850.0, 500.0, 300.0]",2024-05-02 21:00:00,2024-05-02 21:00:00,True,True,True,True,True
2,expand_active_20240520_01,"[cape, d2m, t2m]","[t, u, v]","[850.0, 500.0, 300.0]",2024-05-20 01:00:00,2024-05-20 01:00:00,True,True,True,True,True
3,expand_active_20240523_10,"[cape, d2m, t2m]","[t, u, v]","[850.0, 500.0, 300.0]",2024-05-23 10:00:00,2024-05-23 10:00:00,True,True,True,True,True
4,expand_active_20240609_07,"[cape, d2m, t2m]","[t, u, v]","[850.0, 500.0, 300.0]",2024-06-09 07:00:00,2024-06-09 07:00:00,True,True,True,True,True
5,expand_active_20240613_23,"[cape, d2m, t2m]","[t, u, v]","[850.0, 500.0, 300.0]",2024-06-13 23:00:00,2024-06-13 23:00:00,True,True,True,True,True



All expanded ERA5 inputs satisfy required schema: True


In [9]:
# ------------------------------------------------------------
# Unified ERA5 feature extractor
#
# The primary and expanded ERA5 files use the same required
# variables but differ slightly in extra variables / levels.
#
# This extractor uses only the frozen six-feature specification.
# ERA5 values are valid at the period start time.
# ------------------------------------------------------------


def build_nominal_grid():
    """
    Construct the fixed 0.25-degree coarse-cell centers used by
    the storm-first sample frame.
    """

    grid_lats = np.arange(
        LAT_MIN
        + COARSE_GRID_DEG / 2,
        LAT_MAX,
        COARSE_GRID_DEG,
    )


    grid_lons = np.arange(
        LON_MIN
        + COARSE_GRID_DEG / 2,
        LON_MAX,
        COARSE_GRID_DEG,
    )


    grid_lon_mesh, grid_lat_mesh = np.meshgrid(
        grid_lons,
        grid_lats,
    )


    grid = pd.DataFrame(
        {
            "grid_lat":
                grid_lat_mesh.ravel(),

            "grid_lon":
                grid_lon_mesh.ravel(),
        }
    )


    grid[
        "grid_lat"
    ] = grid[
        "grid_lat"
    ].round(6)


    grid[
        "grid_lon"
    ] = grid[
        "grid_lon"
    ].round(6)


    return grid


NOMINAL_GRID = (
    build_nominal_grid()
)


print(
    "Nominal coarse-grid cells:",
    len(
        NOMINAL_GRID
    ),
)


def extract_era5_features_for_period(
    period_row
):
    """
    Extract the six frozen ERA5 predictors for every nominal
    0.25-degree coarse-grid cell in one period.
    """

    period_id = (
        period_row[
            "period_id"
        ]
    )

    start_utc = pd.Timestamp(
        period_row[
            "start_utc"
        ]
    )

    single_file = (
        period_row[
            "era5_single_file"
        ]
    )

    pressure_file = (
        period_row[
            "era5_pressure_file"
        ]
    )


    with xr.open_dataset(
        single_file
    ) as ds_single, xr.open_dataset(
        pressure_file
    ) as ds_pressure:

        # ----------------------------------------------------
        # Enforce exact temporal alignment.
        # Do not silently choose a nearby ERA5 valid time.
        # ----------------------------------------------------

        single_times = pd.DatetimeIndex(
            pd.to_datetime(
                ds_single[
                    "valid_time"
                ].values
            )
        )

        pressure_times = pd.DatetimeIndex(
            pd.to_datetime(
                ds_pressure[
                    "valid_time"
                ].values
            )
        )


        if start_utc not in single_times:

            raise RuntimeError(
                f"{period_id}: ERA5 single-level file "
                f"does not contain exact start time {start_utc}."
            )


        if start_utc not in pressure_times:

            raise RuntimeError(
                f"{period_id}: ERA5 pressure-level file "
                f"does not contain exact start time {start_utc}."
            )


        single = ds_single.sel(
            valid_time=
                start_utc
        )


        pressure = ds_pressure.sel(
            valid_time=
                start_utc
        )


        # ----------------------------------------------------
        # Required single-level fields
        # ----------------------------------------------------

        t2m = single[
            "t2m"
        ]

        d2m = single[
            "d2m"
        ]

        cape = single[
            "cape"
        ]


        # ----------------------------------------------------
        # Required pressure-level fields
        # ----------------------------------------------------

        temp = pressure[
            "t"
        ]

        u = pressure[
            "u"
        ]

        v = pressure[
            "v"
        ]


        t500 = temp.sel(
            pressure_level=
                500.0
        )


        u850 = u.sel(
            pressure_level=
                850.0
        )

        v850 = v.sel(
            pressure_level=
                850.0
        )


        u500 = u.sel(
            pressure_level=
                500.0
        )

        v500 = v.sel(
            pressure_level=
                500.0
        )


        u300 = u.sel(
            pressure_level=
                300.0
        )

        v300 = v.sel(
            pressure_level=
                300.0
        )


        # ----------------------------------------------------
        # Nominal radar-grid centers
        #
        # Preserve the nearest-grid ERA5 alignment used in the
        # earlier modeling notebook.
        # ----------------------------------------------------

        grid = (
            NOMINAL_GRID
            .copy()
        )


        lat_points = xr.DataArray(
            grid[
                "grid_lat"
            ].to_numpy(),
            dims="sample",
        )


        lon_values = (
            grid[
                "grid_lon"
            ]
            .to_numpy()
            .copy()
        )


        if float(
            single[
                "longitude"
            ].min()
        ) >= 0:

            lon_values = (
                lon_values
                % 360
            )


        lon_points = xr.DataArray(
            lon_values,
            dims="sample",
        )


        def nearest_values(
            da
        ):

            return (
                da
                .sel(
                    latitude=
                        lat_points,

                    longitude=
                        lon_points,

                    method=
                        "nearest",
                )
                .values
                .astype(float)
            )


        u850_v = nearest_values(
            u850
        )

        v850_v = nearest_values(
            v850
        )

        u500_v = nearest_values(
            u500
        )

        v500_v = nearest_values(
            v500
        )

        u300_v = nearest_values(
            u300
        )

        v300_v = nearest_values(
            v300
        )


        result = (
            grid
            .copy()
        )


        result[
            "period_id"
        ] = period_id


        result[
            "era5_valid_time"
        ] = start_utc


        result[
            "cape"
        ] = nearest_values(
            cape
        )


        result[
            "t2m"
        ] = nearest_values(
            t2m
        )


        result[
            "d2m"
        ] = nearest_values(
            d2m
        )


        result[
            "t_500"
        ] = nearest_values(
            t500
        )


        result[
            "shear_850_500"
        ] = np.sqrt(
            (
                u500_v
                -
                u850_v
            ) ** 2
            +
            (
                v500_v
                -
                v850_v
            ) ** 2
        )


        result[
            "shear_850_300"
        ] = np.sqrt(
            (
                u300_v
                -
                u850_v
            ) ** 2
            +
            (
                v300_v
                -
                v850_v
            ) ** 2
        )


    return result


# ------------------------------------------------------------
# One primary + one expanded sanity check
# ------------------------------------------------------------

primary_test_row = (
    PERIOD_PLAN[
        PERIOD_PLAN[
            "panel_group"
        ]
        == "primary"
    ]
    .iloc[0]
)


expanded_test_row = (
    PERIOD_PLAN[
        PERIOD_PLAN[
            "panel_group"
        ]
        == "expanded"
    ]
    .iloc[0]
)


era5_primary_test = (
    extract_era5_features_for_period(
        primary_test_row
    )
)


era5_expanded_test = (
    extract_era5_features_for_period(
        expanded_test_row
    )
)


print(
    "\nPrimary test:",
    primary_test_row[
        "period_id"
    ],
)

print(
    "Rows:",
    len(
        era5_primary_test
    ),
)

print(
    "Missing required values:",
    era5_primary_test[
        ERA5_FEATURES
    ]
    .isna()
    .sum()
    .sum(),
)


print(
    "\nExpanded test:",
    expanded_test_row[
        "period_id"
    ],
)

print(
    "Rows:",
    len(
        era5_expanded_test
    ),
)

print(
    "Missing required values:",
    era5_expanded_test[
        ERA5_FEATURES
    ]
    .isna()
    .sum()
    .sum(),
)


display(
    era5_primary_test[
        [
            "period_id",
            "grid_lat",
            "grid_lon",
        ]
        +
        ERA5_FEATURES
    ]
    .head()
)


display(
    era5_expanded_test[
        [
            "period_id",
            "grid_lat",
            "grid_lon",
        ]
        +
        ERA5_FEATURES
    ]
    .head()
)

Nominal coarse-grid cells: 168

Primary test: active_20240418
Rows: 168
Missing required values: 0

Expanded test: expand_active_20240210_07
Rows: 168
Missing required values: 0


,period_id,grid_lat,grid_lon,cape,t2m,d2m,t_500,shear_850_500,shear_850_300
0,active_20240418,37.625,-91.875,834.875,298.855469,288.686523,261.550537,7.140633,11.403212
1,active_20240418,37.625,-91.625,911.375,298.453125,289.407227,261.941162,6.812204,11.145537
2,active_20240418,37.625,-91.375,1084.125,297.736328,290.329102,262.152100,5.744489,11.053221
3,active_20240418,37.625,-91.125,1246.625,296.656250,291.122070,262.192139,4.518526,10.655042
4,active_20240418,37.625,-90.875,1230.750,295.806641,291.342773,262.064209,4.536932,10.219073


,period_id,grid_lat,grid_lon,cape,t2m,d2m,t_500,shear_850_500,shear_850_300
0,expand_active_20240210_07,37.625,-91.875,0.000,283.905518,282.597168,254.552475,23.026241,54.342682
1,expand_active_20240210_07,37.625,-91.625,0.000,284.712158,283.556152,254.716537,22.857552,54.100479
2,expand_active_20240210_07,37.625,-91.375,5.625,285.382080,284.288574,254.861069,22.356615,53.519076
3,expand_active_20240210_07,37.625,-91.125,12.375,286.098877,284.915527,255.005600,21.316776,52.337940
4,expand_active_20240210_07,37.625,-90.875,29.000,286.561768,285.421387,255.172592,19.429148,50.203517


In [10]:
# ------------------------------------------------------------
# Build ERA5 features for all nine development periods
# ------------------------------------------------------------

era5_period_frames = []


for _, row in (
    PERIOD_PLAN
    .iterrows()
):

    period_id = (
        row[
            "period_id"
        ]
    )


    print(
        "Extracting ERA5:",
        period_id,
    )


    period_features = (
        extract_era5_features_for_period(
            row
        )
    )


    era5_period_frames.append(
        period_features
    )


era5_grid = pd.concat(
    era5_period_frames,
    ignore_index=True,
)


# ------------------------------------------------------------
# Structural audit
# ------------------------------------------------------------

expected_rows = (
    len(
        PERIOD_PLAN
    )
    *
    len(
        NOMINAL_GRID
    )
)


duplicate_keys = (
    era5_grid
    .duplicated(
        subset=[
            "period_id",
            "grid_lat",
            "grid_lon",
        ]
    )
    .sum()
)


missing_required = (
    era5_grid[
        ERA5_FEATURES
    ]
    .isna()
    .sum()
)


era5_period_audit = (
    era5_grid
    .groupby(
        "period_id",
        as_index=False,
    )
    .agg(
        rows=(
            "grid_lat",
            "size",
        ),

        unique_grid_cells=(
            "grid_lat",
            "size",
        ),

        era5_valid_time=(
            "era5_valid_time",
            "first",
        ),
    )
)


era5_period_audit = (
    era5_period_audit
    .merge(
        PERIOD_PLAN[
            [
                "period_id",
                "start_utc",
            ]
        ],
        on="period_id",
        how="left",
        validate="one_to_one",
    )
)


era5_period_audit[
    "time_matches_start"
] = (
    era5_period_audit[
        "era5_valid_time"
    ]
    ==
    era5_period_audit[
        "start_utc"
    ]
)


display(
    era5_period_audit
)


print(
    "\nERA5 GRID AUDIT"
)


print(
    "Rows:",
    len(
        era5_grid
    ),
)


print(
    "Expected rows:",
    expected_rows,
)


print(
    "Periods:",
    era5_grid[
        "period_id"
    ].nunique(),
)


print(
    "Duplicate period-grid keys:",
    duplicate_keys,
)


print(
    "Missing required ERA5 values:",
    int(
        missing_required.sum()
    ),
)


display(
    missing_required
)


print(
    "\nAll periods have 168 grid cells:",
    (
        era5_period_audit[
            "rows"
        ]
        ==
        len(
            NOMINAL_GRID
        )
    )
    .all(),
)


print(
    "All ERA5 valid times match period start:",
    era5_period_audit[
        "time_matches_start"
    ].all(),
)

Extracting ERA5: active_20240418
Extracting ERA5: active_20240508
Extracting ERA5: active_20240526
Extracting ERA5: expand_active_20240210_07
Extracting ERA5: expand_active_20240502_21
Extracting ERA5: expand_active_20240520_01
Extracting ERA5: expand_active_20240523_10
Extracting ERA5: expand_active_20240609_07
Extracting ERA5: expand_active_20240613_23


,period_id,rows,unique_grid_cells,era5_valid_time,start_utc,time_matches_start
0,active_20240418,168,168,2024-04-18 22:00:00,2024-04-18 22:00:00,True
1,active_20240508,168,168,2024-05-08 21:00:00,2024-05-08 21:00:00,True
2,active_20240526,168,168,2024-05-26 22:00:00,2024-05-26 22:00:00,True
3,expand_active_20240210_07,168,168,2024-02-10 07:00:00,2024-02-10 07:00:00,True
4,expand_active_20240502_21,168,168,2024-05-02 21:00:00,2024-05-02 21:00:00,True
5,expand_active_20240520_01,168,168,2024-05-20 01:00:00,2024-05-20 01:00:00,True
6,expand_active_20240523_10,168,168,2024-05-23 10:00:00,2024-05-23 10:00:00,True
7,expand_active_20240609_07,168,168,2024-06-09 07:00:00,2024-06-09 07:00:00,True
8,expand_active_20240613_23,168,168,2024-06-13 23:00:00,2024-06-13 23:00:00,True



ERA5 GRID AUDIT
Rows: 1512
Expected rows: 1512
Periods: 9
Duplicate period-grid keys: 0
Missing required ERA5 values: 0


cape             0
t2m              0
d2m              0
t_500            0
shear_850_500    0
shear_850_300    0
dtype: int64


All periods have 168 grid cells: True
All ERA5 valid times match period start: True


In [11]:
# ------------------------------------------------------------
# Corrected MRMS scan loader
#
# Frozen corrections relative to the exploratory implementation:
#
#   1. both -99 and -999 are treated as unavailable;
#   2. longitude is normalized to [-180, 180];
#   3. the requested analysis domain is applied before later
#      coarse-grid aggregation;
#   4. scan UTC time is parsed directly from the filename.
#
# No storm labels are constructed in this cell.
# ------------------------------------------------------------


def parse_mrms_scan_time(
    path
):
    """
    Parse UTC scan time from an MRMS filename.
    """

    timestamp_match = re.search(
        r"(\d{8}-\d{6})",
        path.name,
    )


    if timestamp_match is None:

        raise ValueError(
            f"Could not parse MRMS timestamp from {path.name}"
        )


    return pd.to_datetime(
        timestamp_match.group(1),
        format="%Y%m%d-%H%M%S",
    )


def ensure_grib2(
    gz_path
):
    """
    Return an uncompressed .grib2 path.

    Existing decompressed files are reused.
    """

    gz_path = Path(
        gz_path
    )


    if gz_path.suffix == ".gz":

        grib_path = (
            gz_path
            .with_suffix("")
        )

    else:

        grib_path = gz_path


    if grib_path.exists():

        return grib_path


    if gz_path.suffix != ".gz":

        raise FileNotFoundError(
            f"Missing MRMS file: {gz_path}"
        )


    with gzip.open(
        gz_path,
        "rb",
    ) as f_in:

        with open(
            grib_path,
            "wb",
        ) as f_out:

            shutil.copyfileobj(
                f_in,
                f_out,
            )


    return grib_path


def load_mrms_scan(
    gz_path
):
    """
    Load one MRMS composite-reflectivity scan over the study domain.

    Returned DataArray:
        dimensions  : latitude, longitude
        longitude   : normalized to [-180, 180]
        missing     : NaN for -99, -999, or non-finite values
    """

    gz_path = Path(
        gz_path
    )


    scan_time = (
        parse_mrms_scan_time(
            gz_path
        )
    )


    grib_path = (
        ensure_grib2(
            gz_path
        )
    )


    ds = xr.open_dataset(
        grib_path,
        engine="cfgrib",
        backend_kwargs={
            "indexpath":
                ""
        },
    )


    try:

        if len(
            ds.data_vars
        ) == 0:

            raise RuntimeError(
                f"No data variables found in {grib_path.name}"
            )


        variable_name = (
            list(
                ds.data_vars
            )[0]
        )


        raw = (
            ds[
                variable_name
            ]
            .squeeze(
                drop=True
            )
        )


        # ----------------------------------------------------
        # Detect longitude convention before spatial subset.
        # ----------------------------------------------------

        longitude_is_360 = (
            float(
                raw[
                    "longitude"
                ].min()
            )
            >= 0
        )


        if longitude_is_360:

            lon_min_select = (
                LON_MIN
                % 360
            )

            lon_max_select = (
                LON_MAX
                % 360
            )

        else:

            lon_min_select = (
                LON_MIN
            )

            lon_max_select = (
                LON_MAX
            )


        # MRMS latitude is normally descending.
        latitude_values = (
            raw[
                "latitude"
            ].values
        )


        if (
            latitude_values[0]
            >
            latitude_values[-1]
        ):

            lat_slice = slice(
                LAT_MAX,
                LAT_MIN,
            )

        else:

            lat_slice = slice(
                LAT_MIN,
                LAT_MAX,
            )


        radar = (
            raw
            .sel(
                latitude=
                    lat_slice,

                longitude=
                    slice(
                        lon_min_select,
                        lon_max_select,
                    ),
            )
            .load()
        )


    finally:

        ds.close()


    # --------------------------------------------------------
    # Frozen missing-data correction
    # --------------------------------------------------------

    valid = (
        np.isfinite(
            radar
        )
        &
        (
            radar
            != -99
        )
        &
        (
            radar
            != -999
        )
    )


    radar = (
        radar
        .where(
            valid
        )
    )


    # --------------------------------------------------------
    # Normalize longitude to [-180, 180]
    # --------------------------------------------------------

    normalized_lon = (
        (
            radar[
                "longitude"
            ]
            +
            180
        )
        % 360
        -
        180
    )


    radar = (
        radar
        .assign_coords(
            longitude=
                normalized_lon
        )
        .sortby(
            "longitude"
        )
        .reset_coords(
            drop=True
        )
    )


    radar = (
        radar
        .expand_dims(
            scan=[
                scan_time
            ]
        )
    )


    return radar


# ------------------------------------------------------------
# One-scan sanity check
#
# Use the newly added cross-midnight June 14 scan.
# ------------------------------------------------------------

mrms_test_row = (
    PERIOD_PLAN[
        PERIOD_PLAN[
            "period_id"
        ]
        ==
        "expand_active_20240613_23"
    ]
    .iloc[0]
)


mrms_test_files = sorted(
    mrms_test_row[
        "radar_dir"
    ]
    .glob(
        "*.grib2.gz"
    )
)


mrms_test_path = (
    mrms_test_files[-1]
)


mrms_test_scan = (
    load_mrms_scan(
        mrms_test_path
    )
)


n_lat = (
    mrms_test_scan
    .sizes[
        "latitude"
    ]
)

n_lon = (
    mrms_test_scan
    .sizes[
        "longitude"
    ]
)


print(
    "Test file:",
    mrms_test_path.name,
)


print(
    "Scan time:",
    pd.Timestamp(
        mrms_test_scan[
            "scan"
        ].values[0]
    ),
)


print(
    "Fine-grid shape:",
    (
        n_lat,
        n_lon,
    ),
)


print(
    "Coarse cells after 25x25 trim:",
    (
        n_lat
        //
        GRID_FACTOR
    )
    *
    (
        n_lon
        //
        GRID_FACTOR
    ),
)


print(
    "Longitude range:",
    (
        float(
            mrms_test_scan[
                "longitude"
            ].min()
        ),
        float(
            mrms_test_scan[
                "longitude"
            ].max()
        ),
    ),
)


print(
    "Latitude range:",
    (
        float(
            mrms_test_scan[
                "latitude"
            ].min()
        ),
        float(
            mrms_test_scan[
                "latitude"
            ].max()
        ),
    ),
)


print(
    "Missing fraction after sentinel masking:",
    float(
        mrms_test_scan
        .isnull()
        .mean()
    ),
)


remaining_bad_sentinels = int(
    (
        (
            mrms_test_scan
            == -99
        )
        |
        (
            mrms_test_scan
            == -999
        )
    )
    .sum()
)


print(
    "Remaining -99/-999 values:",
    remaining_bad_sentinels,
)

ECCODES ERROR   :  Key dataTime (unpack_long): Truncating time: non-zero seconds(39) ignored
ECCODES ERROR   :  Key dataTime (unpack_long): Truncating time: non-zero seconds(39) ignored


Test file: MRMS_MergedReflectivityQCComposite_00.50_20240614-001439.grib2.gz
Scan time: 2024-06-14 00:14:39
Fine-grid shape: (300, 350)
Coarse cells after 25x25 trim: 168
Longitude range: (-91.99500108589166, -88.50500118562223)
Latitude range: (37.50500000000348, 40.49500000000288)
Missing fraction after sentinel masking: 0.6123904761904762
Remaining -99/-999 values: 0


In [ ]:
# ------------------------------------------------------------
# Load one complete MRMS period stack
#
# This cell verifies that all 38 local scans for the repaired
# June 13 / June 14 period can be decoded and concatenated.
# ------------------------------------------------------------


def load_period_radar_stack(
    radar_dir
):
    """
    Decode all local MRMS .grib2.gz scans in one period and
    concatenate them along the scan dimension.
    """

    radar_dir = Path(
        radar_dir
    )


    gz_files = sorted(
        radar_dir.glob(
            "*.grib2.gz"
        )
    )


    if len(
        gz_files
    ) == 0:

        raise FileNotFoundError(
            f"No MRMS .grib2.gz files found in {radar_dir}"
        )


    scan_arrays = []


    for i, gz_path in enumerate(
        gz_files,
        start=1,
    ):

        radar_scan = (
            load_mrms_scan(
                gz_path
            )
        )


        scan_arrays.append(
            radar_scan
        )


        if (
            i == 1
            or
            i % 10 == 0
            or
            i == len(
                gz_files
            )
        ):

            print(
                f"Decoded {i}/{len(gz_files)} scans"
            )


    radar_stack = xr.concat(
        scan_arrays,
        dim="scan",
        coords="minimal",
        compat="override",
        join="exact",
    )


    radar_stack = (
        radar_stack
        .sortby(
            "scan"
        )
    )


    return radar_stack


# ------------------------------------------------------------
# Test the repaired cross-midnight period
# ------------------------------------------------------------

test_period_row = (
    PERIOD_PLAN[
        PERIOD_PLAN[
            "period_id"
        ]
        ==
        "expand_active_20240613_23"
    ]
    .iloc[0]
)


test_radar_stack = (
    load_period_radar_stack(
        test_period_row[
            "radar_dir"
        ]
    )
)


decoded_scan_times = pd.DatetimeIndex(
    pd.to_datetime(
        test_radar_stack[
            "scan"
        ].values
    )
)


print(
    "\nDECODED STACK AUDIT"
)


print(
    "Decoded scans:",
    test_radar_stack.sizes[
        "scan"
    ],
)


print(
    "First decoded scan:",
    decoded_scan_times.min(),
)


print(
    "Last decoded scan:",
    decoded_scan_times.max(),
)


print(
    "Duplicate scan times:",
    int(
        decoded_scan_times
        .duplicated()
        .sum()
    ),
)


print(
    "Fine-grid shape:",
    (
        test_radar_stack.sizes[
            "latitude"
        ],
        test_radar_stack.sizes[
            "longitude"
        ],
    ),
)


decoded_gaps = (
    pd.Series(
        decoded_scan_times
    )
    .diff()
    .dropna()
)


print(
    "Maximum decoded scan gap (min):",
    (
        decoded_gaps.max()
        .total_seconds()
        / 60
    ),
)


print(
    "Overall missing fraction:",
    float(
        test_radar_stack
        .isnull()
        .mean()
    ),
)

In [13]:
# ------------------------------------------------------------
# Corrected 30-minute radar-window summary
#
# Frozen Notebook 03 logic:
#
#   fine-pixel coverage
#       = valid fine pixels / full nominal 625 pixels
#
#   35-dBZ spatial support
#       = pixels >= 35 dBZ / full nominal 625 pixels
#
#   scan-cell eligible
#       = fine-pixel coverage >= 0.80
#
#   window coverage eligible
#       = >= 80% of scans are scan-cell eligible
#
#   strict storm persistence
#       = >= 10% 35-dBZ area in >= 5 scans
#
# MRMS coarse coordinates are snapped to the canonical
# 0.25-degree sample-frame centers before they are used as
# merge keys.
#
# No hail information is used here.
# ------------------------------------------------------------

WINDOW_SCAN_COVERAGE_THRESHOLD = (
    GRID_HOUR_COVERAGE_THRESHOLD
)


def snap_to_nominal_centers(
    values,
    domain_min,
):
    """
    Snap nearly-identical MRMS coarse-grid coordinates to the
    exact canonical 0.25-degree cell centers.

    Example:
        -91.875001 -> -91.875000
    """

    first_center = (
        domain_min
        +
        COARSE_GRID_DEG / 2
    )


    snapped = (
        first_center
        +
        np.round(
            (
                np.asarray(
                    values,
                    dtype=float,
                )
                -
                first_center
            )
            /
            COARSE_GRID_DEG
        )
        *
        COARSE_GRID_DEG
    )


    return np.round(
        snapped,
        6,
    )


def summarize_radar_window(
    radar_stack,
    window_start,
    window_end,
    left_closed,
):
    """
    Summarize one 30-minute radar window on the fixed 0.25-degree grid.

    left_closed=True:
        [window_start, window_end)

    left_closed=False:
        (window_start, window_end]
    """

    scan_times = pd.DatetimeIndex(
        pd.to_datetime(
            radar_stack[
                "scan"
            ].values
        )
    )


    if left_closed:

        keep = (
            (scan_times >= window_start)
            &
            (scan_times < window_end)
        )

    else:

        keep = (
            (scan_times > window_start)
            &
            (scan_times <= window_end)
        )


    selected_times = (
        scan_times[
            keep
        ]
    )


    if len(
        selected_times
    ) == 0:

        raise RuntimeError(
            "No MRMS scans found inside requested radar window."
        )


    window = (
        radar_stack
        .sel(
            scan=
                selected_times
        )
    )


    # --------------------------------------------------------
    # 1. Fine-pixel coverage inside each nominal 25x25 cell
    # --------------------------------------------------------

    valid_pixel_count = (
        window
        .notnull()
        .coarsen(
            latitude=
                GRID_FACTOR,

            longitude=
                GRID_FACTOR,

            boundary=
                "trim",
        )
        .sum()
    )


    pixel_coverage = (
        valid_pixel_count
        /
        NOMINAL_PIXELS_PER_CELL
    )


    scan_cell_eligible = (
        pixel_coverage
        >=
        FINE_PIXEL_COVERAGE_THRESHOLD
    )


    # --------------------------------------------------------
    # 2. Corrected >=35 dBZ spatial support
    #
    # NaN fine pixels contribute zero exceedance pixels, while
    # the denominator remains the full nominal 625 pixels.
    # --------------------------------------------------------

    exceed_pixel_count = (
        (
            window
            >=
            REFLECTIVITY_THRESHOLD_DBZ
        )
        .coarsen(
            latitude=
                GRID_FACTOR,

            longitude=
                GRID_FACTOR,

            boundary=
                "trim",
        )
        .sum()
    )


    support_35 = (
        exceed_pixel_count
        /
        NOMINAL_PIXELS_PER_CELL
    )


    support_35 = (
        support_35
        .where(
            scan_cell_eligible
        )
    )


    # --------------------------------------------------------
    # 3. Coarse-cell reflectivity maximum
    #
    # A scan-cell with <80% fine-pixel coverage is excluded
    # from the cmax summary for that scan.
    # --------------------------------------------------------

    scan_cmax = (
        window
        .coarsen(
            latitude=
                GRID_FACTOR,

            longitude=
                GRID_FACTOR,

            boundary=
                "trim",
        )
        .max(
            skipna=True
        )
        .where(
            scan_cell_eligible
        )
    )


    # --------------------------------------------------------
    # 4. Window-level summaries
    # --------------------------------------------------------

    coverage_fraction = (
        scan_cell_eligible
        .mean(
            dim=
                "scan"
        )
    )


    coverage_eligible = (
        coverage_fraction
        >=
        WINDOW_SCAN_COVERAGE_THRESHOLD
    )


    cmax_dbz = (
        scan_cmax
        .max(
            dim=
                "scan",

            skipna=True,
        )
    )


    n_area5 = (
        (
            support_35
            >=
            0.05
        )
        .sum(
            dim=
                "scan"
        )
    )


    n_area10 = (
        (
            support_35
            >=
            STRICT_AREA_FRACTION
        )
        .sum(
            dim=
                "scan"
        )
    )


    strict_storm = (
        coverage_eligible
        &
        (
            n_area10
            >=
            STRICT_MIN_SCANS
        )
    )


    # --------------------------------------------------------
    # 5. Convert coarse arrays to canonical fixed-grid keys
    # --------------------------------------------------------

    raw_coarse_lats = (
        coverage_fraction[
            "latitude"
        ]
        .values
    )


    raw_coarse_lons = (
        coverage_fraction[
            "longitude"
        ]
        .values
    )


    canonical_lats = (
        snap_to_nominal_centers(
            values=
                raw_coarse_lats,

            domain_min=
                LAT_MIN,
        )
    )


    canonical_lons = (
        snap_to_nominal_centers(
            values=
                raw_coarse_lons,

            domain_min=
                LON_MIN,
        )
    )


    max_lat_snap = float(
        np.max(
            np.abs(
                raw_coarse_lats
                -
                canonical_lats
            )
        )
    )


    max_lon_snap = float(
        np.max(
            np.abs(
                raw_coarse_lons
                -
                canonical_lons
            )
        )
    )


    # This is only intended to remove tiny coordinate offsets,
    # not to repair a genuinely shifted radar grid.
    if (
        max_lat_snap
        >
        0.001
        or
        max_lon_snap
        >
        0.001
    ):

        raise RuntimeError(
            "MRMS coarse grid is not sufficiently close "
            "to the canonical 0.25-degree grid."
        )


    lon_mesh, lat_mesh = np.meshgrid(
        canonical_lons,
        canonical_lats,
    )


    result = pd.DataFrame(
        {
            "grid_lat":
                lat_mesh.ravel(),

            "grid_lon":
                lon_mesh.ravel(),

            "window_n_scans":
                len(
                    selected_times
                ),

            "window_coverage_fraction":
                coverage_fraction
                .values
                .ravel(),

            "window_coverage_eligible":
                coverage_eligible
                .values
                .ravel(),

            "window_cmax_dbz":
                cmax_dbz
                .values
                .ravel(),

            "window_n_area5":
                n_area5
                .values
                .ravel(),

            "window_n_area10":
                n_area10
                .values
                .ravel(),

            "window_strict_storm":
                strict_storm
                .values
                .ravel(),
        }
    )


    return result


# ------------------------------------------------------------
# Sanity check:
# repaired June 13 period, 45/30 FUTURE window
#
# t0         = 23:45
# future     = (23:45, 00:15]
# ------------------------------------------------------------

test_start = pd.Timestamp(
    "2024-06-13 23:00:00"
)


test_t0 = (
    test_start
    +
    pd.Timedelta(
        minutes=45
    )
)


test_target_end = (
    test_t0
    +
    pd.Timedelta(
        minutes=30
    )
)


future_window_test = (
    summarize_radar_window(
        radar_stack=
            test_radar_stack,

        window_start=
            test_t0,

        window_end=
            test_target_end,

        left_closed=
            False,
    )
)


# ------------------------------------------------------------
# Grid-key audit against the nominal 168-cell frame
# ------------------------------------------------------------

actual_keys = set(
    map(
        tuple,
        future_window_test[
            [
                "grid_lat",
                "grid_lon",
            ]
        ]
        .to_numpy()
    )
)


expected_keys = set(
    map(
        tuple,
        NOMINAL_GRID[
            [
                "grid_lat",
                "grid_lon",
            ]
        ]
        .to_numpy()
    )
)


print(
    "Future-window rows:",
    len(
        future_window_test
    ),
)


print(
    "Scans in future window:",
    future_window_test[
        "window_n_scans"
    ].iloc[0],
)


print(
    "Grid exactly matches nominal grid:",
    actual_keys
    ==
    expected_keys,
)


print(
    "Coverage-eligible cells:",
    int(
        future_window_test[
            "window_coverage_eligible"
        ].sum()
    ),
)


print(
    "Strict storm cells:",
    int(
        future_window_test[
            "window_strict_storm"
        ].sum()
    ),
)


print(
    "Coverage fraction range:",
    (
        float(
            future_window_test[
                "window_coverage_fraction"
            ].min()
        ),
        float(
            future_window_test[
                "window_coverage_fraction"
            ].max()
        ),
    ),
)


print(
    "Maximum n_area10:",
    int(
        future_window_test[
            "window_n_area10"
        ].max()
    ),
)


display(
    future_window_test.head()
)

Future-window rows: 168
Scans in future window: 15
Grid exactly matches nominal grid: True
Coverage-eligible cells: 50
Strict storm cells: 17
Coverage fraction range: (0.0, 1.0)
Maximum n_area10: 15


,grid_lat,grid_lon,window_n_scans,window_coverage_fraction,window_coverage_eligible,window_cmax_dbz,window_n_area5,window_n_area10,window_strict_storm
0,40.375,-91.875,15,1.0,True,57.0,15,15,True
1,40.375,-91.625,15,1.0,True,64.0,15,15,True
2,40.375,-91.375,15,1.0,True,50.5,2,1,False
3,40.375,-91.125,15,1.0,True,60.5,8,7,True
4,40.375,-90.875,15,1.0,True,55.0,9,8,True


In [14]:
# ------------------------------------------------------------
# Assemble one complete 45/30 radar regime
#
# Test period:
#   expand_active_20240613_23
#
# Pre-origin predictor window:
#   [23:15, 23:45)
#
# Future storm window:
#   (23:45, 00:15]
#
# The two summaries must map one-to-one onto the same
# canonical 168-cell grid.
# ------------------------------------------------------------

test_pre_start = (
    test_t0
    -
    pd.Timedelta(
        minutes=
            LOOKBACK_MINUTES
    )
)


pre_window_test = (
    summarize_radar_window(
        radar_stack=
            test_radar_stack,

        window_start=
            test_pre_start,

        window_end=
            test_t0,

        left_closed=
            True,
    )
)


# ------------------------------------------------------------
# Rename PRE-origin radar quantities
# ------------------------------------------------------------

pre_window_test = (
    pre_window_test
    .rename(
        columns={
            "window_n_scans":
                "pre_n_scans",

            "window_coverage_fraction":
                "pre_coverage_fraction",

            "window_coverage_eligible":
                "pre_coverage_eligible",

            "window_cmax_dbz":
                "pre_cmax_dbz",

            "window_n_area5":
                "pre_n_area5",

            "window_n_area10":
                "pre_n_area10",

            "window_strict_storm":
                "pre_strict_storm",
        }
    )
)


# ------------------------------------------------------------
# Rename FUTURE radar quantities
#
# future_strict_storm becomes the S+ label.
# ------------------------------------------------------------

future_window_test_named = (
    future_window_test
    .rename(
        columns={
            "window_n_scans":
                "future_n_scans",

            "window_coverage_fraction":
                "future_coverage_fraction",

            "window_coverage_eligible":
                "future_coverage_eligible",

            "window_cmax_dbz":
                "future_cmax_dbz",

            "window_n_area5":
                "future_n_area5",

            "window_n_area10":
                "future_n_area10",

            "window_strict_storm":
                "future_storm",
        }
    )
)


# ------------------------------------------------------------
# One-to-one merge on canonical grid keys
# ------------------------------------------------------------

aligned_4530_test = (
    pre_window_test
    .merge(
        future_window_test_named,
        on=[
            "grid_lat",
            "grid_lon",
        ],
        how="inner",
        validate="one_to_one",
    )
)


aligned_4530_test[
    "period_id"
] = (
    "expand_active_20240613_23"
)


aligned_4530_test[
    "origin_minutes"
] = 45


aligned_4530_test[
    "t0"
] = test_t0


aligned_4530_test[
    "target_end"
] = test_target_end


# ------------------------------------------------------------
# Structural and temporal audit
# ------------------------------------------------------------

duplicate_keys = (
    aligned_4530_test
    .duplicated(
        subset=[
            "period_id",
            "grid_lat",
            "grid_lon",
        ]
    )
    .sum()
)


print(
    "Aligned rows:",
    len(
        aligned_4530_test
    ),
)


print(
    "Duplicate grid keys:",
    duplicate_keys,
)


print(
    "Pre-window scans:",
    aligned_4530_test[
        "pre_n_scans"
    ].unique(),
)


print(
    "Future-window scans:",
    aligned_4530_test[
        "future_n_scans"
    ].unique(),
)


print(
    "Pre coverage-eligible cells:",
    int(
        aligned_4530_test[
            "pre_coverage_eligible"
        ].sum()
    ),
)


print(
    "Future coverage-eligible cells:",
    int(
        aligned_4530_test[
            "future_coverage_eligible"
        ].sum()
    ),
)


print(
    "Future storm cells:",
    int(
        aligned_4530_test[
            "future_storm"
        ].sum()
    ),
)


print(
    "Rows eligible in BOTH pre and future windows:",
    int(
        (
            aligned_4530_test[
                "pre_coverage_eligible"
            ]
            &
            aligned_4530_test[
                "future_coverage_eligible"
            ]
        )
        .sum()
    ),
)


display(
    aligned_4530_test[
        [
            "period_id",
            "origin_minutes",
            "grid_lat",
            "grid_lon",
            "pre_cmax_dbz",
            "pre_n_area5",
            "pre_n_area10",
            "pre_coverage_eligible",
            "future_n_area10",
            "future_coverage_eligible",
            "future_storm",
        ]
    ]
    .head()
)

Aligned rows: 168
Duplicate grid keys: 0
Pre-window scans: [15]
Future-window scans: [15]
Pre coverage-eligible cells: 36
Future coverage-eligible cells: 50
Future storm cells: 17
Rows eligible in BOTH pre and future windows: 36


,period_id,origin_minutes,grid_lat,grid_lon,pre_cmax_dbz,pre_n_area5,pre_n_area10,pre_coverage_eligible,future_n_area10,future_coverage_eligible,future_storm
0,expand_active_20240613_23,45,40.375,-91.875,61.5,15,15,True,15,True,True
1,expand_active_20240613_23,45,40.375,-91.625,60.0,15,15,True,15,True,True
2,expand_active_20240613_23,45,40.375,-91.375,58.5,13,10,True,1,True,False
3,expand_active_20240613_23,45,40.375,-91.125,61.0,12,10,True,7,True,True
4,expand_active_20240613_23,45,40.375,-90.875,50.5,9,9,True,8,True,True


In [ ]:
# ------------------------------------------------------------
# Build temporally aligned radar panels for all
# 9 periods × 2 forecast-origin regimes
#
# Each period is decoded once, then reused for:
#
#   30/30:
#       pre    [start, start+30)
#       future (start+30, start+60]
#
#   45/30:
#       pre    [start+15, start+45)
#       future (start+45, start+75]
#
# No hail labels are added yet.
# ------------------------------------------------------------

import gc


def build_period_radar_regimes(
    period_row
):
    """
    Construct both forecast-origin regimes for one period.
    """

    period_id = (
        period_row[
            "period_id"
        ]
    )

    start_utc = pd.Timestamp(
        period_row[
            "start_utc"
        ]
    )


    print(
        "\n"
        + "=" * 70
    )

    print(
        "Loading radar period:",
        period_id,
    )


    radar_stack = (
        load_period_radar_stack(
            period_row[
                "radar_dir"
            ]
        )
    )


    regime_frames = []


    for origin_minutes in (
        FORECAST_ORIGIN_MINUTES
    ):

        t0 = (
            start_utc
            +
            pd.Timedelta(
                minutes=
                    origin_minutes
            )
        )


        pre_start = (
            t0
            -
            pd.Timedelta(
                minutes=
                    LOOKBACK_MINUTES
            )
        )


        target_end = (
            t0
            +
            pd.Timedelta(
                minutes=
                    TARGET_MINUTES
            )
        )


        # ----------------------------------------------------
        # Pre-origin predictor window
        # ----------------------------------------------------

        pre = (
            summarize_radar_window(
                radar_stack=
                    radar_stack,

                window_start=
                    pre_start,

                window_end=
                    t0,

                left_closed=
                    True,
            )
        )


        pre = (
            pre
            .rename(
                columns={
                    "window_n_scans":
                        "pre_n_scans",

                    "window_coverage_fraction":
                        "pre_coverage_fraction",

                    "window_coverage_eligible":
                        "pre_coverage_eligible",

                    "window_cmax_dbz":
                        "pre_cmax_dbz",

                    "window_n_area5":
                        "pre_n_area5",

                    "window_n_area10":
                        "pre_n_area10",

                    "window_strict_storm":
                        "pre_strict_storm",
                }
            )
        )


        # ----------------------------------------------------
        # Future storm-label window
        # ----------------------------------------------------

        future = (
            summarize_radar_window(
                radar_stack=
                    radar_stack,

                window_start=
                    t0,

                window_end=
                    target_end,

                left_closed=
                    False,
            )
        )


        future = (
            future
            .rename(
                columns={
                    "window_n_scans":
                        "future_n_scans",

                    "window_coverage_fraction":
                        "future_coverage_fraction",

                    "window_coverage_eligible":
                        "future_coverage_eligible",

                    "window_cmax_dbz":
                        "future_cmax_dbz",

                    "window_n_area5":
                        "future_n_area5",

                    "window_n_area10":
                        "future_n_area10",

                    "window_strict_storm":
                        "future_storm",
                }
            )
        )


        # ----------------------------------------------------
        # Exact canonical-grid merge
        # ----------------------------------------------------

        aligned = (
            pre
            .merge(
                future,
                on=[
                    "grid_lat",
                    "grid_lon",
                ],
                how="inner",
                validate="one_to_one",
            )
        )


        aligned[
            "period_id"
        ] = period_id


        aligned[
            "panel_group"
        ] = (
            period_row[
                "panel_group"
            ]
        )


        aligned[
            "period_start_utc"
        ] = start_utc


        aligned[
            "origin_minutes"
        ] = origin_minutes


        aligned[
            "t0"
        ] = t0


        aligned[
            "target_end"
        ] = target_end


        regime_frames.append(
            aligned
        )


        print(
            f"Built {origin_minutes}/30 | "
            f"rows={len(aligned)} | "
            f"pre eligible="
            f"{int(aligned['pre_coverage_eligible'].sum())} | "
            f"future eligible="
            f"{int(aligned['future_coverage_eligible'].sum())} | "
            f"future storms="
            f"{int(aligned['future_storm'].sum())}"
        )


    result = pd.concat(
        regime_frames,
        ignore_index=True,
    )


    del radar_stack
    gc.collect()


    return result


# ------------------------------------------------------------
# Build all nine periods
# ------------------------------------------------------------

radar_period_frames = []


for _, period_row in (
    PERIOD_PLAN
    .iterrows()
):

    radar_period_frames.append(
        build_period_radar_regimes(
            period_row
        )
    )


aligned_radar_all = pd.concat(
    radar_period_frames,
    ignore_index=True,
)


# ------------------------------------------------------------
# Structural audit
# ------------------------------------------------------------

expected_rows = (
    len(
        PERIOD_PLAN
    )
    *
    len(
        FORECAST_ORIGIN_MINUTES
    )
    *
    len(
        NOMINAL_GRID
    )
)


duplicate_keys = (
    aligned_radar_all
    .duplicated(
        subset=[
            "period_id",
            "origin_minutes",
            "grid_lat",
            "grid_lon",
        ]
    )
    .sum()
)


pair_audit = (
    aligned_radar_all
    .groupby(
        [
            "period_id",
            "origin_minutes",
        ],
        as_index=False,
    )
    .agg(
        rows=(
            "grid_lat",
            "size",
        ),

        pre_n_scans=(
            "pre_n_scans",
            "first",
        ),

        future_n_scans=(
            "future_n_scans",
            "first",
        ),

        pre_eligible_cells=(
            "pre_coverage_eligible",
            "sum",
        ),

        future_eligible_cells=(
            "future_coverage_eligible",
            "sum",
        ),

        future_storm_cells=(
            "future_storm",
            "sum",
        ),
    )
)


pair_audit[
    "both_window_eligible_cells"
] = (
    aligned_radar_all
    .assign(
        both_eligible=
            (
                aligned_radar_all[
                    "pre_coverage_eligible"
                ]
                &
                aligned_radar_all[
                    "future_coverage_eligible"
                ]
            )
    )
    .groupby(
        [
            "period_id",
            "origin_minutes",
        ]
    )[
        "both_eligible"
    ]
    .sum()
    .to_numpy()
)


display(
    pair_audit
)


print(
    "\nFULL RADAR PANEL AUDIT"
)


print(
    "Rows:",
    len(
        aligned_radar_all
    ),
)


print(
    "Expected rows:",
    expected_rows,
)


print(
    "Period-origin pairs:",
    pair_audit.shape[0],
)


print(
    "Duplicate period-origin-grid keys:",
    duplicate_keys,
)


print(
    "All pairs have 168 grid cells:",
    (
        pair_audit[
            "rows"
        ]
        ==
        len(
            NOMINAL_GRID
        )
    )
    .all(),
)


print(
    "All pre windows have 15 scans:",
    (
        pair_audit[
            "pre_n_scans"
        ]
        ==
        15
    )
    .all(),
)


print(
    "All future windows have 15 scans:",
    (
        pair_audit[
            "future_n_scans"
        ]
        ==
        15
    )
    .all(),
)

In [16]:
# ------------------------------------------------------------
# Clean NOAA hail observations and convert local event time
# to UTC using the explicit CZ_TIMEZONE offset.
#
# No storm information is used here.
# No forecast-window labels are constructed yet.
# ------------------------------------------------------------

hail_raw = pd.read_csv(
    HAIL_PATH,
    low_memory=False,
)


required_hail_columns = {
    "BEGIN_DATE_TIME",
    "BEGIN_LAT",
    "BEGIN_LON",
    "CZ_TIMEZONE",
}


missing_hail_columns = (
    required_hail_columns
    -
    set(
        hail_raw.columns
    )
)


if missing_hail_columns:

    raise RuntimeError(
        "Missing required NOAA hail columns: "
        f"{sorted(missing_hail_columns)}"
    )


hail_2024 = (
    hail_raw
    .copy()
)


# ------------------------------------------------------------
# Numeric coordinates
# ------------------------------------------------------------

hail_2024[
    "BEGIN_LAT"
] = pd.to_numeric(
    hail_2024[
        "BEGIN_LAT"
    ],
    errors="coerce",
)


hail_2024[
    "BEGIN_LON"
] = pd.to_numeric(
    hail_2024[
        "BEGIN_LON"
    ],
    errors="coerce",
)


# ------------------------------------------------------------
# Local event time
# ------------------------------------------------------------

hail_2024[
    "begin_local"
] = pd.to_datetime(
    hail_2024[
        "BEGIN_DATE_TIME"
    ],
    errors="coerce",
)


# ------------------------------------------------------------
# Parse NOAA timezone offset
#
# Examples:
#     CST-6 -> -6
#     EST-5 -> -5
#
# NOAA local time satisfies:
#
#     local = UTC + offset
#
# therefore:
#
#     UTC = local - offset
# ------------------------------------------------------------

hail_2024[
    "utc_offset_hours"
] = pd.to_numeric(
    hail_2024[
        "CZ_TIMEZONE"
    ]
    .astype(str)
    .str.extract(
        r"([+-]\d+(?:\.\d+)?)$",
        expand=False,
    ),
    errors="coerce",
)


hail_2024[
    "begin_utc"
] = (
    hail_2024[
        "begin_local"
    ]
    -
    pd.to_timedelta(
        hail_2024[
            "utc_offset_hours"
        ],
        unit="h",
    )
)


# ------------------------------------------------------------
# Keep valid 2024 reports inside the fixed study domain
# ------------------------------------------------------------

hail_2024 = (
    hail_2024[
        hail_2024[
            "begin_utc"
        ].notna()
        &
        hail_2024[
            "BEGIN_LAT"
        ].notna()
        &
        hail_2024[
            "BEGIN_LON"
        ].notna()
        &
        (
            hail_2024[
                "begin_utc"
            ].dt.year
            ==
            2024
        )
        &
        hail_2024[
            "BEGIN_LAT"
        ].between(
            LAT_MIN,
            LAT_MAX,
            inclusive="both",
        )
        &
        hail_2024[
            "BEGIN_LON"
        ].between(
            LON_MIN,
            LON_MAX,
            inclusive="both",
        )
    ]
    .copy()
    .sort_values(
        "begin_utc"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

print(
    "2024 pilot-domain NOAA hail reports:",
    len(
        hail_2024
    ),
)


print(
    "UTC time range:",
    hail_2024[
        "begin_utc"
    ].min(),
    "->",
    hail_2024[
        "begin_utc"
    ].max(),
)


print(
    "\nTimezone counts:"
)


display(
    hail_2024[
        "CZ_TIMEZONE"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "Missing UTC timestamps:",
    int(
        hail_2024[
            "begin_utc"
        ]
        .isna()
        .sum()
    ),
)


print(
    "Latitude range:",
    (
        float(
            hail_2024[
                "BEGIN_LAT"
            ].min()
        ),
        float(
            hail_2024[
                "BEGIN_LAT"
            ].max()
        ),
    ),
)


print(
    "Longitude range:",
    (
        float(
            hail_2024[
                "BEGIN_LON"
            ].min()
        ),
        float(
            hail_2024[
                "BEGIN_LON"
            ].max()
        ),
    ),
)

2024 pilot-domain NOAA hail reports: 247
UTC time range: 2024-02-08 22:45:00 -> 2024-10-04 11:30:00

Timezone counts:


CZ_TIMEZONE
CST-6    247
Name: count, dtype: int64

Missing UTC timestamps: 0
Latitude range: (37.54, 40.48)
Longitude range: (-91.9802, -88.5186)


In [17]:
# ------------------------------------------------------------
# Future hail overlay and H1S0 structural audit
#
# For each period-origin pair:
#
#   H+ = 1
#       if at least one NOAA hail report occurs in
#       (t0, t0 + 30 min]
#       and maps to that canonical 0.25-degree grid cell.
#
# Hail is overlaid AFTER the hail-independent radar storm
# construction.
#
# No H=1, S=0 cases are removed or relabeled.
# ------------------------------------------------------------


def assign_hail_to_nominal_grid(
    hail_df
):
    """
    Assign NOAA hail reports to the nearest canonical
    0.25-degree grid-cell center.
    """

    assigned = (
        hail_df
        .copy()
    )


    if len(
        assigned
    ) == 0:

        assigned[
            "grid_lat"
        ] = pd.Series(
            dtype=float
        )

        assigned[
            "grid_lon"
        ] = pd.Series(
            dtype=float
        )

        return assigned


    grid_lats = np.sort(
        NOMINAL_GRID[
            "grid_lat"
        ]
        .unique()
    )


    grid_lons = np.sort(
        NOMINAL_GRID[
            "grid_lon"
        ]
        .unique()
    )


    assigned[
        "grid_lat"
    ] = assigned[
        "BEGIN_LAT"
    ].apply(
        lambda x:
            grid_lats[
                np.argmin(
                    np.abs(
                        grid_lats
                        -
                        x
                    )
                )
            ]
    )


    assigned[
        "grid_lon"
    ] = assigned[
        "BEGIN_LON"
    ].apply(
        lambda x:
            grid_lons[
                np.argmin(
                    np.abs(
                        grid_lons
                        -
                        x
                    )
                )
            ]
    )


    assigned[
        "grid_lat"
    ] = assigned[
        "grid_lat"
    ].round(6)


    assigned[
        "grid_lon"
    ] = assigned[
        "grid_lon"
    ].round(6)


    return assigned


# ------------------------------------------------------------
# Build H+ labels for all 18 period-origin pairs
# ------------------------------------------------------------

hail_label_rows = []

hail_window_audit_rows = []


for (
    period_id,
    origin_minutes
), group in (
    aligned_radar_all
    .groupby(
        [
            "period_id",
            "origin_minutes",
        ],
        sort=False,
    )
):

    t0_values = (
        group[
            "t0"
        ]
        .drop_duplicates()
    )


    target_end_values = (
        group[
            "target_end"
        ]
        .drop_duplicates()
    )


    if (
        len(
            t0_values
        )
        != 1
        or
        len(
            target_end_values
        )
        != 1
    ):

        raise RuntimeError(
            f"{period_id} {origin_minutes}/30 "
            "does not have unique t0 / target_end."
        )


    t0 = pd.Timestamp(
        t0_values.iloc[0]
    )


    target_end = pd.Timestamp(
        target_end_values.iloc[0]
    )


    # --------------------------------------------------------
    # Exact future hail window:
    #
    #     (t0, target_end]
    # --------------------------------------------------------

    future_hail_reports = (
        hail_2024[
            (
                hail_2024[
                    "begin_utc"
                ]
                >
                t0
            )
            &
            (
                hail_2024[
                    "begin_utc"
                ]
                <=
                target_end
            )
        ]
        .copy()
    )


    assigned_hail = (
        assign_hail_to_nominal_grid(
            future_hail_reports
        )
    )


    hail_cells = (
        assigned_hail[
            [
                "grid_lat",
                "grid_lon",
            ]
        ]
        .drop_duplicates()
        .copy()
    )


    hail_cells[
        "period_id"
    ] = period_id


    hail_cells[
        "origin_minutes"
    ] = origin_minutes


    hail_cells[
        "future_hail"
    ] = 1


    hail_label_rows.append(
        hail_cells
    )


    hail_window_audit_rows.append(
        {
            "period_id":
                period_id,

            "origin_minutes":
                origin_minutes,

            "t0":
                t0,

            "target_end":
                target_end,

            "hail_reports":
                len(
                    future_hail_reports
                ),

            "hail_grid_cells":
                len(
                    hail_cells
                ),
        }
    )


hail_window_audit = pd.DataFrame(
    hail_window_audit_rows
)


if len(
    hail_label_rows
) > 0:

    hail_labels = pd.concat(
        hail_label_rows,
        ignore_index=True,
    )

else:

    hail_labels = pd.DataFrame(
        columns=[
            "period_id",
            "origin_minutes",
            "grid_lat",
            "grid_lon",
            "future_hail",
        ]
    )


# ------------------------------------------------------------
# Overlay H+ onto the complete radar panel
# ------------------------------------------------------------

future_panel = (
    aligned_radar_all
    .merge(
        hail_labels,
        on=[
            "period_id",
            "origin_minutes",
            "grid_lat",
            "grid_lon",
        ],
        how="left",
        validate="one_to_one",
    )
)


future_panel[
    "future_hail"
] = (
    future_panel[
        "future_hail"
    ]
    .fillna(0)
    .astype(int)
)


future_panel[
    "future_storm"
] = (
    future_panel[
        "future_storm"
    ]
    .astype(int)
)


future_panel[
    "both_coverage_eligible"
] = (
    future_panel[
        "pre_coverage_eligible"
    ]
    &
    future_panel[
        "future_coverage_eligible"
    ]
)


# ------------------------------------------------------------
# Modeling panel:
#
# Both predictor-window radar and target-window radar must
# satisfy the frozen coverage criterion.
#
# Importantly, hail-positive rows failing coverage are counted
# below before exclusion.
# ------------------------------------------------------------

modeling_panel_radar = (
    future_panel[
        future_panel[
            "both_coverage_eligible"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Structural audit by period-origin pair
# ------------------------------------------------------------

structural_rows = []


for (
    period_id,
    origin_minutes
), group in (
    future_panel
    .groupby(
        [
            "period_id",
            "origin_minutes",
        ],
        sort=False,
    )
):

    eligible = (
        group[
            group[
                "both_coverage_eligible"
            ]
        ]
    )


    hail_all = int(
        group[
            "future_hail"
        ].sum()
    )


    hail_eligible = int(
        eligible[
            "future_hail"
        ].sum()
    )


    h1s0 = int(
        (
            (
                eligible[
                    "future_hail"
                ]
                ==
                1
            )
            &
            (
                eligible[
                    "future_storm"
                ]
                ==
                0
            )
        )
        .sum()
    )


    h1s1 = int(
        (
            (
                eligible[
                    "future_hail"
                ]
                ==
                1
            )
            &
            (
                eligible[
                    "future_storm"
                ]
                ==
                1
            )
        )
        .sum()
    )


    structural_rows.append(
        {
            "period_id":
                period_id,

            "origin_minutes":
                origin_minutes,

            "all_grid_rows":
                len(
                    group
                ),

            "both_eligible_rows":
                len(
                    eligible
                ),

            "future_storm_cells":
                int(
                    eligible[
                        "future_storm"
                    ].sum()
                ),

            "hail_cells_all_grid":
                hail_all,

            "hail_cells_both_eligible":
                hail_eligible,

            "hail_cells_excluded_by_coverage":
                (
                    hail_all
                    -
                    hail_eligible
                ),

            "H1S1":
                h1s1,

            "H1S0":
                h1s0,

            "hail_capture_if_eligible":
                (
                    h1s1
                    /
                    hail_eligible
                    if hail_eligible
                    >
                    0
                    else np.nan
                ),
        }
    )


structural_audit = pd.DataFrame(
    structural_rows
)


display(
    hail_window_audit
)


print(
    "\nSTRUCTURAL HAIL-STORM AUDIT"
)


display(
    structural_audit
)


print(
    "\nTOTALS"
)


print(
    "Hail reports across all future windows:",
    int(
        hail_window_audit[
            "hail_reports"
        ].sum()
    ),
)


print(
    "Hail-positive grid cells before coverage filtering:",
    int(
        future_panel[
            "future_hail"
        ].sum()
    ),
)


print(
    "Hail-positive grid cells retained in modeling panel:",
    int(
        modeling_panel_radar[
            "future_hail"
        ].sum()
    ),
)


print(
    "Hail-positive grid cells excluded by coverage:",
    int(
        future_panel[
            "future_hail"
        ].sum()
        -
        modeling_panel_radar[
            "future_hail"
        ].sum()
    ),
)


total_h1s0 = int(
    (
        (
            modeling_panel_radar[
                "future_hail"
            ]
            ==
            1
        )
        &
        (
            modeling_panel_radar[
                "future_storm"
            ]
            ==
            0
        )
    )
    .sum()
)


total_h1s1 = int(
    (
        (
            modeling_panel_radar[
                "future_hail"
            ]
            ==
            1
        )
        &
        (
            modeling_panel_radar[
                "future_storm"
            ]
            ==
            1
        )
    )
    .sum()
)


print(
    "Eligible H=1, S=1:",
    total_h1s1,
)


print(
    "Eligible H=1, S=0:",
    total_h1s0,
)


if (
    modeling_panel_radar[
        "future_hail"
    ].sum()
    >
    0
):

    print(
        "Eligible hail capture by storm proxy:",
        total_h1s1
        /
        modeling_panel_radar[
            "future_hail"
        ].sum(),
    )

,period_id,origin_minutes,t0,target_end,hail_reports,hail_grid_cells
0,active_20240418,30,2024-04-18 22:30:00,2024-04-18 23:00:00,3,1
1,active_20240418,45,2024-04-18 22:45:00,2024-04-18 23:15:00,4,3
2,active_20240508,30,2024-05-08 21:30:00,2024-05-08 22:00:00,2,2
3,active_20240508,45,2024-05-08 21:45:00,2024-05-08 22:15:00,1,1
4,active_20240526,30,2024-05-26 22:30:00,2024-05-26 23:00:00,6,5
5,active_20240526,45,2024-05-26 22:45:00,2024-05-26 23:15:00,5,5
6,expand_active_20240210_07,30,2024-02-10 07:30:00,2024-02-10 08:00:00,0,0
7,expand_active_20240210_07,45,2024-02-10 07:45:00,2024-02-10 08:15:00,0,0
8,expand_active_20240502_21,30,2024-05-02 21:30:00,2024-05-02 22:00:00,3,1
9,expand_active_20240502_21,45,2024-05-02 21:45:00,2024-05-02 22:15:00,4,1



STRUCTURAL HAIL-STORM AUDIT


,period_id,origin_minutes,all_grid_rows,both_eligible_rows,future_storm_cells,hail_cells_all_grid,hail_cells_both_eligible,hail_cells_excluded_by_coverage,H1S1,H1S0,hail_capture_if_eligible
0,active_20240418,30,168,52,31,1,0,1,0,0,NaN
1,active_20240418,45,168,48,28,3,2,1,2,0,1.0
2,active_20240508,30,168,43,28,2,2,0,2,0,1.0
3,active_20240508,45,168,44,29,1,0,1,0,0,NaN
4,active_20240526,30,168,51,34,5,4,1,4,0,1.0
5,active_20240526,45,168,64,39,5,5,0,5,0,1.0
6,expand_active_20240210_07,30,168,2,2,0,0,0,0,0,NaN
7,expand_active_20240210_07,45,168,1,1,0,0,0,0,0,NaN
8,expand_active_20240502_21,30,168,28,21,1,0,1,0,0,NaN
9,expand_active_20240502_21,45,168,40,24,1,0,1,0,0,NaN



TOTALS
Hail reports across all future windows: 37
Hail-positive grid cells before coverage filtering: 25
Hail-positive grid cells retained in modeling panel: 17
Hail-positive grid cells excluded by coverage: 8
Eligible H=1, S=1: 17
Eligible H=1, S=0: 0
Eligible hail capture by storm proxy: 1.0


In [18]:
# ------------------------------------------------------------
# Assemble the final temporally aligned modeling panel
#
# Population:
#   rows passing BOTH
#       pre-origin radar coverage
#       future-target radar coverage
#
# Predictors:
#   6 ERA5 environmental predictors
#   3 pre-t0 radar predictors
#
# Targets:
#   future_storm = S+
#   future_hail  = H+
#
# No post-t0 radar quantity is used as a predictor.
# ------------------------------------------------------------


final_modeling_panel = (
    modeling_panel_radar
    .merge(
        era5_grid,
        on=[
            "period_id",
            "grid_lat",
            "grid_lon",
        ],
        how="left",
        validate="many_to_one",
    )
)


# ------------------------------------------------------------
# Explicit predictor leakage audit
# ------------------------------------------------------------

expected_features = [
    "cape",
    "t2m",
    "d2m",
    "t_500",
    "shear_850_500",
    "shear_850_300",
    "pre_cmax_dbz",
    "pre_n_area5",
    "pre_n_area10",
]


if (
    FINAL_FEATURES
    !=
    expected_features
):

    raise RuntimeError(
        "FINAL_FEATURES no longer matches the frozen "
        "nine-feature specification."
    )


missing_features = (
    final_modeling_panel[
        FINAL_FEATURES
    ]
    .isna()
    .sum()
)


duplicate_keys = (
    final_modeling_panel
    .duplicated(
        subset=[
            "period_id",
            "origin_minutes",
            "grid_lat",
            "grid_lon",
        ]
    )
    .sum()
)


# ------------------------------------------------------------
# Period × regime modeling audit
# ------------------------------------------------------------

modeling_audit = (
    final_modeling_panel
    .groupby(
        [
            "period_id",
            "origin_minutes",
        ],
        as_index=False,
    )
    .agg(
        rows=(
            "future_hail",
            "size",
        ),

        storm_positives=(
            "future_storm",
            "sum",
        ),

        hail_positives=(
            "future_hail",
            "sum",
        ),
    )
)


display(
    modeling_audit
)


print(
    "\nFINAL MODELING PANEL AUDIT"
)


print(
    "Rows:",
    len(
        final_modeling_panel
    ),
)


print(
    "Periods:",
    final_modeling_panel[
        "period_id"
    ].nunique(),
)


print(
    "Period-origin pairs:",
    modeling_audit.shape[0],
)


print(
    "Duplicate period-origin-grid keys:",
    duplicate_keys,
)


print(
    "Missing values in frozen predictors:",
    int(
        missing_features.sum()
    ),
)


display(
    missing_features
)


print(
    "\nBY FORECAST REGIME"
)


regime_audit = (
    final_modeling_panel
    .groupby(
        "origin_minutes",
        as_index=False,
    )
    .agg(
        rows=(
            "future_hail",
            "size",
        ),

        storm_positives=(
            "future_storm",
            "sum",
        ),

        hail_positives=(
            "future_hail",
            "sum",
        ),

        periods=(
            "period_id",
            "nunique",
        ),
    )
)


display(
    regime_audit
)


print(
    "\nTOTAL STORM POSITIVES:",
    int(
        final_modeling_panel[
            "future_storm"
        ].sum()
    ),
)


print(
    "TOTAL HAIL POSITIVES:",
    int(
        final_modeling_panel[
            "future_hail"
        ].sum()
    ),
)


print(
    "\nFrozen predictors:"
)


for feature in (
    FINAL_FEATURES
):

    print(
        " -",
        feature,
    )

,period_id,origin_minutes,rows,storm_positives,hail_positives
0,active_20240418,30,52,31,0
1,active_20240418,45,48,28,2
2,active_20240508,30,43,28,2
3,active_20240508,45,44,29,0
4,active_20240526,30,51,34,4
5,active_20240526,45,64,39,5
6,expand_active_20240210_07,30,2,2,0
7,expand_active_20240210_07,45,1,1,0
8,expand_active_20240502_21,30,28,21,0
9,expand_active_20240502_21,45,40,24,0



FINAL MODELING PANEL AUDIT
Rows: 486
Periods: 9
Period-origin pairs: 18
Duplicate period-origin-grid keys: 0
Missing values in frozen predictors: 0


cape             0
t2m              0
d2m              0
t_500            0
shear_850_500    0
shear_850_300    0
pre_cmax_dbz     0
pre_n_area5      0
pre_n_area10     0
dtype: int64


BY FORECAST REGIME


,origin_minutes,rows,storm_positives,hail_positives,periods
0,30,232,148,8,9
1,45,254,149,9,9



TOTAL STORM POSITIVES: 297
TOTAL HAIL POSITIVES: 17

Frozen predictors:
 - cape
 - t2m
 - d2m
 - t_500
 - shear_850_500
 - shear_850_300
 - pre_cmax_dbz
 - pre_n_area5
 - pre_n_area10


In [19]:
# ------------------------------------------------------------
# Leave-one-period-out feasibility audit
#
# Models will be evaluated separately for:
#
#   30/30
#   45/30
#
# For each held-out period:
#
#   Direct:
#       train H+ on all eligible training rows
#
#   Stage 1:
#       train S+ on all eligible training rows
#
#   Stage 2:
#       train H+ only among training rows with S+ = 1
#
# This cell does NOT fit any models.
# ------------------------------------------------------------

lopo_audit_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    regime_df = (
        final_modeling_panel[
            final_modeling_panel[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .copy()
    )


    periods = sorted(
        regime_df[
            "period_id"
        ]
        .unique()
    )


    for test_period in periods:

        train = (
            regime_df[
                regime_df[
                    "period_id"
                ]
                !=
                test_period
            ]
            .copy()
        )


        test = (
            regime_df[
                regime_df[
                    "period_id"
                ]
                ==
                test_period
            ]
            .copy()
        )


        stage2_train = (
            train[
                train[
                    "future_storm"
                ]
                ==
                1
            ]
            .copy()
        )


        direct_two_classes = (
            train[
                "future_hail"
            ]
            .nunique()
            ==
            2
        )


        stage1_two_classes = (
            train[
                "future_storm"
            ]
            .nunique()
            ==
            2
        )


        stage2_two_classes = (
            stage2_train[
                "future_hail"
            ]
            .nunique()
            ==
            2
        )


        fit_feasible = (
            direct_two_classes
            and
            stage1_two_classes
            and
            stage2_two_classes
        )


        lopo_audit_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "test_period":
                    test_period,

                "train_rows":
                    len(
                        train
                    ),

                "test_rows":
                    len(
                        test
                    ),

                "train_hail_pos":
                    int(
                        train[
                            "future_hail"
                        ].sum()
                    ),

                "test_hail_pos":
                    int(
                        test[
                            "future_hail"
                        ].sum()
                    ),

                "train_storm_pos":
                    int(
                        train[
                            "future_storm"
                        ].sum()
                    ),

                "test_storm_pos":
                    int(
                        test[
                            "future_storm"
                        ].sum()
                    ),

                "stage2_train_rows":
                    len(
                        stage2_train
                    ),

                "stage2_train_hail_pos":
                    int(
                        stage2_train[
                            "future_hail"
                        ].sum()
                    ),

                "stage2_train_hail_neg":
                    int(
                        (
                            stage2_train[
                                "future_hail"
                            ]
                            ==
                            0
                        )
                        .sum()
                    ),

                "direct_two_classes":
                    direct_two_classes,

                "stage1_two_classes":
                    stage1_two_classes,

                "stage2_two_classes":
                    stage2_two_classes,

                "fit_feasible":
                    fit_feasible,
            }
        )


lopo_audit = pd.DataFrame(
    lopo_audit_rows
)


display(
    lopo_audit
)


print(
    "\nLOPO FEASIBILITY AUDIT"
)


print(
    "Folds:",
    len(
        lopo_audit
    ),
)


print(
    "30/30 folds:",
    int(
        (
            lopo_audit[
                "origin_minutes"
            ]
            ==
            30
        )
        .sum()
    ),
)


print(
    "45/30 folds:",
    int(
        (
            lopo_audit[
                "origin_minutes"
            ]
            ==
            45
        )
        .sum()
    ),
)


print(
    "All direct training folds have both hail classes:",
    lopo_audit[
        "direct_two_classes"
    ].all(),
)


print(
    "All Stage-1 training folds have both storm classes:",
    lopo_audit[
        "stage1_two_classes"
    ].all(),
)


print(
    "All Stage-2 training folds have both hail classes:",
    lopo_audit[
        "stage2_two_classes"
    ].all(),
)


print(
    "All folds fit-feasible:",
    lopo_audit[
        "fit_feasible"
    ].all(),
)


print(
    "\nMinimum training hail positives:",
    int(
        lopo_audit[
            "train_hail_pos"
        ].min()
    ),
)


print(
    "Minimum Stage-2 hail positives:",
    int(
        lopo_audit[
            "stage2_train_hail_pos"
        ].min()
    ),
)


print(
    "Minimum Stage-2 hail negatives:",
    int(
        lopo_audit[
            "stage2_train_hail_neg"
        ].min()
    ),
)

,origin_minutes,test_period,train_rows,test_rows,train_hail_pos,test_hail_pos,train_storm_pos,test_storm_pos,stage2_train_rows,stage2_train_hail_pos,stage2_train_hail_neg,direct_two_classes,stage1_two_classes,stage2_two_classes,fit_feasible
0,30,active_20240418,180,52,8,0,117,31,117,8,109,True,True,True,True
1,30,active_20240508,189,43,6,2,120,28,120,6,114,True,True,True,True
2,30,active_20240526,181,51,4,4,114,34,114,4,110,True,True,True,True
3,30,expand_active_20240210_07,230,2,8,0,146,2,146,8,138,True,True,True,True
4,30,expand_active_20240502_21,204,28,8,0,127,21,127,8,119,True,True,True,True
5,30,expand_active_20240520_01,231,1,8,0,147,1,147,8,139,True,True,True,True
6,30,expand_active_20240523_10,216,16,8,0,135,13,135,8,127,True,True,True,True
7,30,expand_active_20240609_07,225,7,7,1,145,3,145,7,138,True,True,True,True
8,30,expand_active_20240613_23,200,32,7,1,133,15,133,7,126,True,True,True,True
9,45,active_20240418,206,48,7,2,121,28,121,7,114,True,True,True,True



LOPO FEASIBILITY AUDIT
Folds: 18
30/30 folds: 9
45/30 folds: 9
All direct training folds have both hail classes: True
All Stage-1 training folds have both storm classes: True
All Stage-2 training folds have both hail classes: True
All folds fit-feasible: True

Minimum training hail positives: 4
Minimum Stage-2 hail positives: 4
Minimum Stage-2 hail negatives: 106


In [20]:
# ------------------------------------------------------------
# Corrected same-X Direct vs Hierarchical LOPO comparison
#
# Direct:
#     P(H+ | X-)
#
# Stage 1:
#     P(S+ | X-)
#
# Stage 2:
#     P(H+ | S+=1, X-)
#
# Hierarchical:
#     P(S+ | X-) * P(H+ | S+=1, X-)
#
# All three learners use the same frozen nine predictors.
# No post-t0 variable is used as a predictor.
# ------------------------------------------------------------

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


def make_logistic():
    """
    Frozen simple learner used for the primary comparison.
    """

    return Pipeline(
        [
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "model",
                LogisticRegression(
                    C=1.0,
                    penalty="l2",
                    solver="liblinear",
                    max_iter=5000,
                    random_state=20260913,
                ),
            ),
        ]
    )


# ------------------------------------------------------------
# LOPO predictions
# ------------------------------------------------------------

oof_rows = []

model_fold_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    regime_df = (
        final_modeling_panel[
            final_modeling_panel[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .copy()
    )


    periods = sorted(
        regime_df[
            "period_id"
        ]
        .unique()
    )


    for test_period in periods:

        train = (
            regime_df[
                regime_df[
                    "period_id"
                ]
                !=
                test_period
            ]
            .copy()
        )


        test = (
            regime_df[
                regime_df[
                    "period_id"
                ]
                ==
                test_period
            ]
            .copy()
        )


        stage2_train = (
            train[
                train[
                    "future_storm"
                ]
                ==
                1
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Direct hail model
        # ----------------------------------------------------

        direct_model = (
            make_logistic()
        )


        direct_model.fit(
            train[
                FINAL_FEATURES
            ],
            train[
                "future_hail"
            ],
        )


        p_direct = (
            direct_model
            .predict_proba(
                test[
                    FINAL_FEATURES
                ]
            )[:, 1]
        )


        # ----------------------------------------------------
        # Stage 1:
        # future storm occurrence
        # ----------------------------------------------------

        stage1_model = (
            make_logistic()
        )


        stage1_model.fit(
            train[
                FINAL_FEATURES
            ],
            train[
                "future_storm"
            ],
        )


        p_stage1 = (
            stage1_model
            .predict_proba(
                test[
                    FINAL_FEATURES
                ]
            )[:, 1]
        )


        # ----------------------------------------------------
        # Stage 2:
        # future hail conditional on future storm
        # ----------------------------------------------------

        stage2_model = (
            make_logistic()
        )


        stage2_model.fit(
            stage2_train[
                FINAL_FEATURES
            ],
            stage2_train[
                "future_hail"
            ],
        )


        p_stage2 = (
            stage2_model
            .predict_proba(
                test[
                    FINAL_FEATURES
                ]
            )[:, 1]
        )


        # ----------------------------------------------------
        # Soft hierarchical probability
        # ----------------------------------------------------

        p_hierarchical = (
            p_stage1
            *
            p_stage2
        )


        # ----------------------------------------------------
        # Store one OOF row per held-out grid cell
        # ----------------------------------------------------

        fold_predictions = pd.DataFrame(
            {
                "period_id":
                    test[
                        "period_id"
                    ].to_numpy(),

                "origin_minutes":
                    test[
                        "origin_minutes"
                    ].to_numpy(),

                "grid_lat":
                    test[
                        "grid_lat"
                    ].to_numpy(),

                "grid_lon":
                    test[
                        "grid_lon"
                    ].to_numpy(),

                "future_hail":
                    test[
                        "future_hail"
                    ].to_numpy(
                        dtype=int
                    ),

                "future_storm":
                    test[
                        "future_storm"
                    ].to_numpy(
                        dtype=int
                    ),

                "p_direct":
                    p_direct,

                "p_stage1":
                    p_stage1,

                "p_stage2":
                    p_stage2,

                "p_hierarchical":
                    p_hierarchical,
            }
        )


        oof_rows.append(
            fold_predictions
        )


        model_fold_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "test_period":
                    test_period,

                "train_rows":
                    len(
                        train
                    ),

                "test_rows":
                    len(
                        test
                    ),

                "train_hail_pos":
                    int(
                        train[
                            "future_hail"
                        ].sum()
                    ),

                "stage2_train_rows":
                    len(
                        stage2_train
                    ),

                "stage2_train_hail_pos":
                    int(
                        stage2_train[
                            "future_hail"
                        ].sum()
                    ),

                "test_hail_pos":
                    int(
                        test[
                            "future_hail"
                        ].sum()
                    ),
            }
        )


samex_oof = pd.concat(
    oof_rows,
    ignore_index=True,
)


samex_fold_audit = pd.DataFrame(
    model_fold_rows
)


# ------------------------------------------------------------
# Prediction integrity audit
# ------------------------------------------------------------

oof_duplicate_keys = (
    samex_oof
    .duplicated(
        subset=[
            "period_id",
            "origin_minutes",
            "grid_lat",
            "grid_lon",
        ]
    )
    .sum()
)


probability_columns = [
    "p_direct",
    "p_stage1",
    "p_stage2",
    "p_hierarchical",
]


probabilities_finite = (
    np.isfinite(
        samex_oof[
            probability_columns
        ]
        .to_numpy()
    )
    .all()
)


probabilities_in_range = (
    (
        samex_oof[
            probability_columns
        ]
        >=
        0
    )
    &
    (
        samex_oof[
            probability_columns
        ]
        <=
        1
    )
).all().all()


print(
    "OOF rows:",
    len(
        samex_oof
    ),
)


print(
    "Expected OOF rows:",
    len(
        final_modeling_panel
    ),
)


print(
    "Duplicate OOF keys:",
    oof_duplicate_keys,
)


print(
    "All probabilities finite:",
    probabilities_finite,
)


print(
    "All probabilities in [0, 1]:",
    probabilities_in_range,
)


# ------------------------------------------------------------
# Pooled OOF scoring by forecast regime
# ------------------------------------------------------------

def score_hail_predictions(
    y,
    p
):
    """
    Score pooled hail probabilities.

    Higher:
        PR-AUC
        ROC-AUC

    Lower:
        Brier
        Log loss
    """

    y = np.asarray(
        y,
        dtype=int,
    )


    p = np.clip(
        np.asarray(
            p,
            dtype=float,
        ),
        1e-8,
        1 - 1e-8,
    )


    return {
        "n":
            len(
                y
            ),

        "positives":
            int(
                y.sum()
            ),

        "prevalence":
            float(
                y.mean()
            ),

        "pr_auc":
            float(
                average_precision_score(
                    y,
                    p,
                )
            ),

        "roc_auc":
            float(
                roc_auc_score(
                    y,
                    p,
                )
            ),

        "brier":
            float(
                brier_score_loss(
                    y,
                    p,
                )
            ),

        "log_loss":
            float(
                log_loss(
                    y,
                    p,
                    labels=[
                        0,
                        1,
                    ],
                )
            ),
    }


pooled_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    regime_oof = (
        samex_oof[
            samex_oof[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .copy()
    )


    for (
        model_name,
        probability_column
    ) in [
        (
            "Direct same-X",
            "p_direct",
        ),
        (
            "Hierarchical same-X",
            "p_hierarchical",
        ),
    ]:

        scores = (
            score_hail_predictions(
                y=
                    regime_oof[
                        "future_hail"
                    ],

                p=
                    regime_oof[
                        probability_column
                    ],
            )
        )


        pooled_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "model":
                    model_name,

                **scores,
            }
        )


samex_pooled_results = pd.DataFrame(
    pooled_rows
)


display(
    samex_pooled_results
)


# ------------------------------------------------------------
# Paired metric differences
#
# Positive delta always means hierarchy is better.
# ------------------------------------------------------------

comparison_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    this = (
        samex_pooled_results[
            samex_pooled_results[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .set_index(
            "model"
        )
    )


    direct = this.loc[
        "Direct same-X"
    ]


    hierarchy = this.loc[
        "Hierarchical same-X"
    ]


    comparison_rows.append(
        {
            "origin_minutes":
                origin_minutes,

            "delta_pr":
                (
                    hierarchy[
                        "pr_auc"
                    ]
                    -
                    direct[
                        "pr_auc"
                    ]
                ),

            "delta_roc":
                (
                    hierarchy[
                        "roc_auc"
                    ]
                    -
                    direct[
                        "roc_auc"
                    ]
                ),

            # Positive = hierarchy lower / better
            "delta_brier":
                (
                    direct[
                        "brier"
                    ]
                    -
                    hierarchy[
                        "brier"
                    ]
                ),

            # Positive = hierarchy lower / better
            "delta_log_loss":
                (
                    direct[
                        "log_loss"
                    ]
                    -
                    hierarchy[
                        "log_loss"
                    ]
                ),
        }
    )


samex_comparison = pd.DataFrame(
    comparison_rows
)


print(
    "\nPAIRED POOLED DIFFERENCES"
)


print(
    "Positive delta = hierarchy better."
)


display(
    samex_comparison
)

OOF rows: 486
Expected OOF rows: 486
Duplicate OOF keys: 0
All probabilities finite: True
All probabilities in [0, 1]: True


,origin_minutes,model,n,positives,prevalence,pr_auc,roc_auc,brier,log_loss
0,30,Direct same-X,232,8,0.034483,0.048532,0.584821,0.044308,0.182178
1,30,Hierarchical same-X,232,8,0.034483,0.059260,0.660156,0.039289,0.164625
2,45,Direct same-X,254,9,0.035433,0.051851,0.597732,0.040872,0.173344
3,45,Hierarchical same-X,254,9,0.035433,0.093064,0.682993,0.035246,0.152728



PAIRED POOLED DIFFERENCES
Positive delta = hierarchy better.


,origin_minutes,delta_pr,delta_roc,delta_brier,delta_log_loss
0,30,0.010727,0.075335,0.005018,0.017553
1,45,0.041212,0.085261,0.005626,0.020616


In [21]:
# ------------------------------------------------------------
# Leakage-free LOPO prevalence baseline
#
# For each held-out period, predict every test row using the
# hail prevalence estimated ONLY from the other eight periods.
#
# This is a no-feature, intercept-only benchmark.
#
# It should NOT be interpreted as a long-term climatological
# hail probability because the nine periods are event-enriched.
# ------------------------------------------------------------

baseline_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    regime_df = (
        final_modeling_panel[
            final_modeling_panel[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .copy()
    )


    for test_period in sorted(
        regime_df[
            "period_id"
        ]
        .unique()
    ):

        train = (
            regime_df[
                regime_df[
                    "period_id"
                ]
                !=
                test_period
            ]
        )


        train_hail_prevalence = float(
            train[
                "future_hail"
            ]
            .mean()
        )


        baseline_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "period_id":
                    test_period,

                "p_climatology":
                    train_hail_prevalence,
            }
        )


# Internal variable name retained for downstream compatibility.
# Public-facing terminology is "LOPO prevalence baseline".
lopo_climatology = pd.DataFrame(
    baseline_rows
)


samex_oof_with_baseline = (
    samex_oof
    .merge(
        lopo_climatology,
        on=[
            "origin_minutes",
            "period_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


if (
    samex_oof_with_baseline[
        "p_climatology"
    ]
    .isna()
    .any()
):

    raise RuntimeError(
        "Missing LOPO prevalence-baseline predictions."
    )


# ------------------------------------------------------------
# Score prevalence baseline, Direct, and Hierarchical
# ------------------------------------------------------------

baseline_comparison_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    regime_oof = (
        samex_oof_with_baseline[
            samex_oof_with_baseline[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .copy()
    )


    for (
        model_name,
        probability_column
    ) in [
        (
            "LOPO prevalence baseline",
            "p_climatology",
        ),
        (
            "Direct same-X",
            "p_direct",
        ),
        (
            "Hierarchical same-X",
            "p_hierarchical",
        ),
    ]:

        scores = (
            score_hail_predictions(
                y=
                    regime_oof[
                        "future_hail"
                    ],

                p=
                    regime_oof[
                        probability_column
                    ],
            )
        )


        baseline_comparison_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "model":
                    model_name,

                **scores,
            }
        )


baseline_comparison = pd.DataFrame(
    baseline_comparison_rows
)


display(
    baseline_comparison
)


# ------------------------------------------------------------
# Skill relative to leakage-free prevalence baseline
# ------------------------------------------------------------

skill_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    this = (
        baseline_comparison[
            baseline_comparison[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .set_index(
            "model"
        )
    )


    baseline = (
        this.loc[
            "LOPO prevalence baseline"
        ]
    )


    for model_name in [
        "Direct same-X",
        "Hierarchical same-X",
    ]:

        model = (
            this.loc[
                model_name
            ]
        )


        skill_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "model":
                    model_name,

                "delta_pr_vs_climatology":
                    (
                        model[
                            "pr_auc"
                        ]
                        -
                        baseline[
                            "pr_auc"
                        ]
                    ),

                "delta_roc_vs_climatology":
                    (
                        model[
                            "roc_auc"
                        ]
                        -
                        baseline[
                            "roc_auc"
                        ]
                    ),

                "brier_skill_score":
                    (
                        1
                        -
                        model[
                            "brier"
                        ]
                        /
                        baseline[
                            "brier"
                        ]
                    ),

                "delta_log_loss_vs_climatology":
                    (
                        baseline[
                            "log_loss"
                        ]
                        -
                        model[
                            "log_loss"
                        ]
                    ),
            }
        )


# Internal variable name retained temporarily for downstream
# compatibility. Public-facing terminology is prevalence baseline.
skill_vs_climatology = pd.DataFrame(
    skill_rows
)


print(
    "\nSKILL RELATIVE TO LOPO PREVALENCE BASELINE"
)


print(
    "Positive values = model better than the "
    "held-out training-prevalence baseline."
)


display(
    skill_vs_climatology
)

,origin_minutes,model,n,positives,prevalence,pr_auc,roc_auc,brier,log_loss
0,30,LOPO prevalence baseline,232,8,0.034483,0.025291,0.225167,0.033834,0.158761
1,30,Direct same-X,232,8,0.034483,0.048532,0.584821,0.044308,0.182178
2,30,Hierarchical same-X,232,8,0.034483,0.059260,0.660156,0.039289,0.164625
3,45,LOPO prevalence baseline,254,9,0.035433,0.026529,0.235601,0.034750,0.162824
4,45,Direct same-X,254,9,0.035433,0.051851,0.597732,0.040872,0.173344
5,45,Hierarchical same-X,254,9,0.035433,0.093064,0.682993,0.035246,0.152728



SKILL RELATIVE TO LOPO PREVALENCE BASELINE
Positive values = model better than the held-out training-prevalence baseline.


,origin_minutes,model,delta_pr_vs_climatology,delta_roc_vs_climatology,brier_skill_score,delta_log_loss_vs_climatology
0,30,Direct same-X,0.023242,0.359654,-0.309552,-0.023417
1,30,Hierarchical same-X,0.033969,0.434989,-0.161231,-0.005864
2,45,Direct same-X,0.025322,0.362132,-0.176181,-0.010520
3,45,Hierarchical same-X,0.066535,0.447392,-0.014283,0.010096


In [22]:
# ------------------------------------------------------------
# Period-cluster bootstrap for paired Direct vs Hierarchical
# differences.
#
# Resampling unit:
#     period_id
#
# NOT individual grid cells.
#
# This measures sensitivity to the composition of the nine
# sampled development periods. It does NOT establish external
# climatological generalization.
# ------------------------------------------------------------

N_BOOTSTRAP = 5000

BOOTSTRAP_SEED = 20260913


rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


bootstrap_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    regime_oof = (
        samex_oof[
            samex_oof[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .copy()
    )


    periods = np.array(
        sorted(
            regime_oof[
                "period_id"
            ]
            .unique()
        )
    )


    period_groups = {
        period_id:
            regime_oof[
                regime_oof[
                    "period_id"
                ]
                ==
                period_id
            ]
            .copy()

        for period_id in periods
    }


    valid_replicates = 0
    skipped_no_positive = 0


    for bootstrap_id in range(
        N_BOOTSTRAP
    ):

        sampled_periods = (
            rng.choice(
                periods,
                size=len(
                    periods
                ),
                replace=True,
            )
        )


        boot = pd.concat(
            [
                period_groups[
                    period_id
                ]

                for period_id
                in sampled_periods
            ],
            ignore_index=True,
        )


        y = (
            boot[
                "future_hail"
            ]
            .to_numpy(
                dtype=int
            )
        )


        # PR-AUC and especially ROC-AUC are not meaningful
        # for a bootstrap replicate with no hail positives.
        if (
            np.unique(
                y
            ).size
            <
            2
        ):

            skipped_no_positive += 1

            continue


        direct_scores = (
            score_hail_predictions(
                y=
                    y,

                p=
                    boot[
                        "p_direct"
                    ],
            )
        )


        hierarchy_scores = (
            score_hail_predictions(
                y=
                    y,

                p=
                    boot[
                        "p_hierarchical"
                    ],
            )
        )


        bootstrap_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "bootstrap_id":
                    bootstrap_id,

                "delta_pr":
                    (
                        hierarchy_scores[
                            "pr_auc"
                        ]
                        -
                        direct_scores[
                            "pr_auc"
                        ]
                    ),

                "delta_roc":
                    (
                        hierarchy_scores[
                            "roc_auc"
                        ]
                        -
                        direct_scores[
                            "roc_auc"
                        ]
                    ),

                # Positive = hierarchy better
                "delta_brier":
                    (
                        direct_scores[
                            "brier"
                        ]
                        -
                        hierarchy_scores[
                            "brier"
                        ]
                    ),

                # Positive = hierarchy better
                "delta_log_loss":
                    (
                        direct_scores[
                            "log_loss"
                        ]
                        -
                        hierarchy_scores[
                            "log_loss"
                        ]
                    ),

                "hail_positives":
                    int(
                        y.sum()
                    ),
            }
        )


        valid_replicates += 1


    print(
        f"{origin_minutes}/30 | "
        f"valid={valid_replicates} | "
        f"skipped_no_positive={skipped_no_positive}"
    )


period_bootstrap = pd.DataFrame(
    bootstrap_rows
)


# ------------------------------------------------------------
# Summarize paired bootstrap distributions
# ------------------------------------------------------------

summary_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    boot = (
        period_bootstrap[
            period_bootstrap[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
    )


    point = (
        samex_comparison[
            samex_comparison[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .iloc[0]
    )


    for metric in [
        "delta_pr",
        "delta_roc",
        "delta_brier",
        "delta_log_loss",
    ]:

        values = (
            boot[
                metric
            ]
            .dropna()
            .to_numpy()
        )


        summary_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "metric":
                    metric,

                "point_estimate":
                    float(
                        point[
                            metric
                        ]
                    ),

                "bootstrap_median":
                    float(
                        np.median(
                            values
                        )
                    ),

                "q025":
                    float(
                        np.quantile(
                            values,
                            0.025,
                        )
                    ),

                "q975":
                    float(
                        np.quantile(
                            values,
                            0.975,
                        )
                    ),

                "fraction_delta_gt_0":
                    float(
                        np.mean(
                            values
                            >
                            0
                        )
                    ),

                "valid_replicates":
                    len(
                        values
                    ),
            }
        )


bootstrap_summary = pd.DataFrame(
    summary_rows
)


print(
    "\nPERIOD-CLUSTER BOOTSTRAP SUMMARY"
)


print(
    "Positive delta = hierarchy better."
)


display(
    bootstrap_summary
)

30/30 | valid=4973 | skipped_no_positive=27
45/30 | valid=4967 | skipped_no_positive=33

PERIOD-CLUSTER BOOTSTRAP SUMMARY
Positive delta = hierarchy better.


,origin_minutes,metric,point_estimate,bootstrap_median,q025,q975,fraction_delta_gt_0,valid_replicates
0,30,delta_pr,0.010727,0.013710,-0.018258,0.092596,0.817816,4973
1,30,delta_roc,0.075335,0.079111,-0.148108,0.113459,0.877539,4973
2,30,delta_brier,0.005018,0.004804,0.000417,0.011600,0.989946,4973
3,30,delta_log_loss,0.017553,0.017373,0.005017,0.032969,0.997989,4973
4,45,delta_pr,0.041212,0.053579,-0.000581,0.186346,0.964969,4967
5,45,delta_roc,0.085261,0.076190,-0.017479,0.177439,0.952084,4967
6,45,delta_brier,0.005626,0.005347,0.000005,0.018302,0.975035,4967
7,45,delta_log_loss,0.020616,0.019579,-0.002794,0.066539,0.920475,4967


In [23]:
# ------------------------------------------------------------
# Stage-level diagnostic:
#
#   Stage 1:
#       P(S+ | X-)
#
#   Stage 2:
#       P(H+ | S+=1, X-)
#
# Goal:
#   identify where predictive information enters the
#   storm -> hail chain.
#
# All metrics use held-out LOPO predictions.
# ------------------------------------------------------------


# ------------------------------------------------------------
# Build leakage-free stage-specific prevalence baselines
#
# Stage 1 baseline:
#     training storm prevalence
#
# Stage 2 baseline:
#     training hail prevalence among training S+=1 rows
#
# These are training-prevalence baselines, not long-term
# climatological probabilities.
# ------------------------------------------------------------

stage_baseline_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    regime_df = (
        final_modeling_panel[
            final_modeling_panel[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .copy()
    )


    for test_period in sorted(
        regime_df[
            "period_id"
        ]
        .unique()
    ):

        train = (
            regime_df[
                regime_df[
                    "period_id"
                ]
                !=
                test_period
            ]
            .copy()
        )


        stage2_train = (
            train[
                train[
                    "future_storm"
                ]
                ==
                1
            ]
            .copy()
        )


        stage_baseline_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "period_id":
                    test_period,

                "p_stage1_climatology":
                    float(
                        train[
                            "future_storm"
                        ].mean()
                    ),

                "p_stage2_climatology":
                    float(
                        stage2_train[
                            "future_hail"
                        ].mean()
                    ),
            }
        )


stage_baselines = pd.DataFrame(
    stage_baseline_rows
)


stage_oof = (
    samex_oof
    .merge(
        stage_baselines,
        on=[
            "origin_minutes",
            "period_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


if (
    stage_oof[
        [
            "p_stage1_climatology",
            "p_stage2_climatology",
        ]
    ]
    .isna()
    .any()
    .any()
):

    raise RuntimeError(
        "Missing stage-specific prevalence-baseline predictions."
    )


# ------------------------------------------------------------
# Score Stage 1 and Stage 2
# ------------------------------------------------------------

stage_score_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    regime_oof = (
        stage_oof[
            stage_oof[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Stage 1:
    # evaluate on the full eligible held-out population
    # --------------------------------------------------------

    for (
        model_name,
        probability_column
    ) in [
        (
            "Stage-1 LOPO prevalence baseline",
            "p_stage1_climatology",
        ),
        (
            "Stage-1 model",
            "p_stage1",
        ),
    ]:

        scores = (
            score_hail_predictions(
                y=
                    regime_oof[
                        "future_storm"
                    ],

                p=
                    regime_oof[
                        probability_column
                    ],
            )
        )


        stage_score_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "stage":
                    "Stage 1: storm",

                "model":
                    model_name,

                **scores,
            }
        )


    # --------------------------------------------------------
    # Stage 2:
    # evaluate ONLY on held-out rows where S+ = 1
    # --------------------------------------------------------

    storm_test = (
        regime_oof[
            regime_oof[
                "future_storm"
            ]
            ==
            1
        ]
        .copy()
    )


    for (
        model_name,
        probability_column
    ) in [
        (
            "Stage-2 LOPO prevalence baseline",
            "p_stage2_climatology",
        ),
        (
            "Stage-2 model",
            "p_stage2",
        ),
    ]:

        scores = (
            score_hail_predictions(
                y=
                    storm_test[
                        "future_hail"
                    ],

                p=
                    storm_test[
                        probability_column
                    ],
            )
        )


        stage_score_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "stage":
                    "Stage 2: hail | storm",

                "model":
                    model_name,

                **scores,
            }
        )


stage_scores = pd.DataFrame(
    stage_score_rows
)


display(
    stage_scores
)


# ------------------------------------------------------------
# Stage-specific skill relative to the leakage-free
# prevalence baseline
# ------------------------------------------------------------

stage_skill_rows = []


for (
    origin_minutes,
    stage
), group in (
    stage_scores
    .groupby(
        [
            "origin_minutes",
            "stage",
        ],
        sort=False,
    )
):

    group = (
        group
        .set_index(
            "model"
        )
    )


    if (
        stage
        ==
        "Stage 1: storm"
    ):

        baseline_name = (
            "Stage-1 LOPO prevalence baseline"
        )

        model_name = (
            "Stage-1 model"
        )

    else:

        baseline_name = (
            "Stage-2 LOPO prevalence baseline"
        )

        model_name = (
            "Stage-2 model"
        )


    baseline = (
        group.loc[
            baseline_name
        ]
    )


    model = (
        group.loc[
            model_name
        ]
    )


    stage_skill_rows.append(
        {
            "origin_minutes":
                origin_minutes,

            "stage":
                stage,

            "n":
                int(
                    model[
                        "n"
                    ]
                ),

            "positives":
                int(
                    model[
                        "positives"
                    ]
                ),

            "prevalence":
                float(
                    model[
                        "prevalence"
                    ]
                ),

            "delta_pr_vs_climatology":
                (
                    model[
                        "pr_auc"
                    ]
                    -
                    baseline[
                        "pr_auc"
                    ]
                ),

            "delta_roc_vs_climatology":
                (
                    model[
                        "roc_auc"
                    ]
                    -
                    baseline[
                        "roc_auc"
                    ]
                ),

            "brier_skill_score":
                (
                    1
                    -
                    model[
                        "brier"
                    ]
                    /
                    baseline[
                        "brier"
                    ]
                ),

            "delta_log_loss_vs_climatology":
                (
                    baseline[
                        "log_loss"
                    ]
                    -
                    model[
                        "log_loss"
                    ]
                ),
        }
    )


# Internal column names are retained temporarily for downstream
# compatibility. Public-facing terminology is prevalence baseline.
stage_skill = pd.DataFrame(
    stage_skill_rows
)


print(
    "\nSTAGE-SPECIFIC SKILL"
)


print(
    "Positive values = model better than "
    "its leakage-free stage-specific prevalence baseline."
)


display(
    stage_skill
)

,origin_minutes,stage,model,n,positives,prevalence,pr_auc,roc_auc,brier,log_loss
0,30,Stage 1: storm,Stage-1 LOPO prevalence baseline,232,148,0.637931,0.578505,0.393420,0.233607,0.660340
1,30,Stage 1: storm,Stage-1 model,232,148,0.637931,0.768168,0.766651,0.176490,0.662354
2,30,Stage 2: hail | storm,Stage-2 LOPO prevalence baseline,148,8,0.054054,0.039676,0.219643,0.052438,0.224001
3,30,Stage 2: hail | storm,Stage-2 model,148,8,0.054054,0.081773,0.617857,0.061170,0.233864
4,45,Stage 1: storm,Stage-1 LOPO prevalence baseline,254,149,0.586614,0.546242,0.436274,0.243802,0.680761
5,45,Stage 1: storm,Stage-1 model,254,149,0.586614,0.801182,0.770789,0.188212,0.568247
6,45,Stage 2: hail | storm,Stage-2 LOPO prevalence baseline,149,9,0.060403,0.045234,0.228571,0.058483,0.245222
7,45,Stage 2: hail | storm,Stage-2 model,149,9,0.060403,0.083721,0.547619,0.071052,0.269239



STAGE-SPECIFIC SKILL
Positive values = model better than its leakage-free stage-specific prevalence baseline.


,origin_minutes,stage,n,positives,prevalence,delta_pr_vs_climatology,delta_roc_vs_climatology,brier_skill_score,delta_log_loss_vs_climatology
0,30,Stage 1: storm,232,148,0.637931,0.189663,0.373230,0.244503,-0.002013
1,30,Stage 2: hail | storm,148,8,0.054054,0.042098,0.398214,-0.166536,-0.009863
2,45,Stage 1: storm,254,149,0.586614,0.254940,0.334516,0.228013,0.112514
3,45,Stage 2: hail | storm,149,9,0.060403,0.038487,0.319048,-0.214927,-0.024017


In [24]:
# ------------------------------------------------------------
# Hierarchy component ablation
#
# Goal:
# determine whether the hierarchy's predictive improvement
# comes primarily from Stage 1, Stage 2, or both.
#
# All component baselines are leakage-free training-prevalence
# estimates from the corresponding LOPO training folds.
#
# Definitions:
#
#   Direct:
#       P(H+ | X-)
#
#   Stage-1-only hierarchy:
#       P(S+ | X-) *
#       training prevalence P(H+ | S+=1)
#
#   Stage-2-only hierarchy:
#       training prevalence P(S+) *
#       P(H+ | S+=1, X-)
#
#   Full hierarchy:
#       P(S+ | X-) *
#       P(H+ | S+=1, X-)
# ------------------------------------------------------------


component_oof = (
    stage_oof
    .copy()
)


# ------------------------------------------------------------
# Construct component probabilities
# ------------------------------------------------------------

component_oof[
    "p_stage1_only_hierarchy"
] = (
    component_oof[
        "p_stage1"
    ]
    *
    component_oof[
        "p_stage2_climatology"
    ]
)


component_oof[
    "p_stage2_only_hierarchy"
] = (
    component_oof[
        "p_stage1_climatology"
    ]
    *
    component_oof[
        "p_stage2"
    ]
)


# Internal variable name retained for compatibility.
# This is the product of two LOPO training-prevalence baselines.
component_oof[
    "p_climatology_product"
] = (
    component_oof[
        "p_stage1_climatology"
    ]
    *
    component_oof[
        "p_stage2_climatology"
    ]
)


# ------------------------------------------------------------
# Probability integrity audit
# ------------------------------------------------------------

component_probability_columns = [
    "p_climatology_product",
    "p_direct",
    "p_stage1_only_hierarchy",
    "p_stage2_only_hierarchy",
    "p_hierarchical",
]


print(
    "All component probabilities finite:",
    np.isfinite(
        component_oof[
            component_probability_columns
        ]
        .to_numpy()
    )
    .all(),
)


print(
    "All component probabilities in [0, 1]:",
    (
        (
            component_oof[
                component_probability_columns
            ]
            >=
            0
        )
        &
        (
            component_oof[
                component_probability_columns
            ]
            <=
            1
        )
    )
    .all()
    .all(),
)


# ------------------------------------------------------------
# Score all component formulations
# ------------------------------------------------------------

component_score_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    regime_oof = (
        component_oof[
            component_oof[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .copy()
    )


    formulations = [
        (
            "Prevalence-baseline product",
            "p_climatology_product",
        ),
        (
            "Direct same-X",
            "p_direct",
        ),
        (
            "Stage-1-only hierarchy",
            "p_stage1_only_hierarchy",
        ),
        (
            "Stage-2-only hierarchy",
            "p_stage2_only_hierarchy",
        ),
        (
            "Full hierarchy",
            "p_hierarchical",
        ),
    ]


    for (
        model_name,
        probability_column
    ) in formulations:

        scores = (
            score_hail_predictions(
                y=
                    regime_oof[
                        "future_hail"
                    ],

                p=
                    regime_oof[
                        probability_column
                    ],
            )
        )


        component_score_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "model":
                    model_name,

                **scores,
            }
        )


component_scores = pd.DataFrame(
    component_score_rows
)


display(
    component_scores
)


# ------------------------------------------------------------
# Difference relative to Direct
#
# Positive = formulation better than Direct
# ------------------------------------------------------------

component_delta_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    this = (
        component_scores[
            component_scores[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .set_index(
            "model"
        )
    )


    direct = (
        this.loc[
            "Direct same-X"
        ]
    )


    for model_name in [
        "Stage-1-only hierarchy",
        "Stage-2-only hierarchy",
        "Full hierarchy",
    ]:

        model = (
            this.loc[
                model_name
            ]
        )


        component_delta_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "model":
                    model_name,

                "delta_pr_vs_direct":
                    (
                        model[
                            "pr_auc"
                        ]
                        -
                        direct[
                            "pr_auc"
                        ]
                    ),

                "delta_roc_vs_direct":
                    (
                        model[
                            "roc_auc"
                        ]
                        -
                        direct[
                            "roc_auc"
                        ]
                    ),

                # Positive = lower / better Brier
                "delta_brier_vs_direct":
                    (
                        direct[
                            "brier"
                        ]
                        -
                        model[
                            "brier"
                        ]
                    ),

                # Positive = lower / better log loss
                "delta_log_loss_vs_direct":
                    (
                        direct[
                            "log_loss"
                        ]
                        -
                        model[
                            "log_loss"
                        ]
                    ),
            }
        )


component_deltas = pd.DataFrame(
    component_delta_rows
)


print(
    "\nCOMPONENT GAINS RELATIVE TO DIRECT"
)


print(
    "Positive values = component formulation better than Direct."
)


display(
    component_deltas
)

All component probabilities finite: True
All component probabilities in [0, 1]: True


,origin_minutes,model,n,positives,prevalence,pr_auc,roc_auc,brier,log_loss
0,30,Prevalence-baseline product,232,8,0.034483,0.025291,0.225167,0.033834,0.158761
1,30,Direct same-X,232,8,0.034483,0.048532,0.584821,0.044308,0.182178
2,30,Stage-1-only hierarchy,232,8,0.034483,0.036540,0.493862,0.033942,0.157633
3,30,Stage-2-only hierarchy,232,8,0.034483,0.054668,0.629464,0.037875,0.162947
4,30,Full hierarchy,232,8,0.034483,0.059260,0.660156,0.039289,0.164625
5,45,Prevalence-baseline product,254,9,0.035433,0.026529,0.235601,0.034750,0.162824
6,45,Direct same-X,254,9,0.035433,0.051851,0.597732,0.040872,0.173344
7,45,Stage-1-only hierarchy,254,9,0.035433,0.040633,0.554649,0.034260,0.150270
8,45,Stage-2-only hierarchy,254,9,0.035433,0.051940,0.594104,0.039060,0.170468
9,45,Full hierarchy,254,9,0.035433,0.093064,0.682993,0.035246,0.152728



COMPONENT GAINS RELATIVE TO DIRECT
Positive values = component formulation better than Direct.


,origin_minutes,model,delta_pr_vs_direct,delta_roc_vs_direct,delta_brier_vs_direct,delta_log_loss_vs_direct
0,30,Stage-1-only hierarchy,-0.011992,-0.090960,0.010366,0.024544
1,30,Stage-2-only hierarchy,0.006136,0.044643,0.006433,0.019231
2,30,Full hierarchy,0.010727,0.075335,0.005018,0.017553
3,45,Stage-1-only hierarchy,-0.011219,-0.043084,0.006612,0.023074
4,45,Stage-2-only hierarchy,0.000088,-0.003628,0.001813,0.002876
5,45,Full hierarchy,0.041212,0.085261,0.005626,0.020616


In [25]:
# ------------------------------------------------------------
# OOF calibration audit
#
# Hail positives are sparse (8 and 9 by regime), so this cell
# deliberately avoids fitting a calibration model or reporting
# unstable calibration slopes.
#
# We inspect:
#
#   1. observed prevalence
#   2. mean predicted probability
#   3. calibration-in-the-large error
#   4. coarse 4-bin reliability summaries
#
# All predictions are held-out LOPO predictions.
#
# The no-feature benchmark is the LOPO training-prevalence
# baseline, not a long-term climatological hail probability.
# ------------------------------------------------------------

CALIBRATION_BINS = 4


calibration_rows = []
reliability_rows = []


for origin_minutes in (
    FORECAST_ORIGIN_MINUTES
):

    regime_oof = (
        samex_oof_with_baseline[
            samex_oof_with_baseline[
                "origin_minutes"
            ]
            ==
            origin_minutes
        ]
        .copy()
    )


    model_columns = [
        (
            "LOPO prevalence baseline",
            "p_climatology",
        ),
        (
            "Direct same-X",
            "p_direct",
        ),
        (
            "Hierarchical same-X",
            "p_hierarchical",
        ),
    ]


    for (
        model_name,
        probability_column
    ) in model_columns:

        y = (
            regime_oof[
                "future_hail"
            ]
            .to_numpy(
                dtype=int
            )
        )


        p = (
            regime_oof[
                probability_column
            ]
            .to_numpy(
                dtype=float
            )
        )


        observed_prevalence = float(
            y.mean()
        )


        mean_prediction = float(
            p.mean()
        )


        calibration_rows.append(
            {
                "origin_minutes":
                    origin_minutes,

                "model":
                    model_name,

                "n":
                    len(
                        y
                    ),

                "positives":
                    int(
                        y.sum()
                    ),

                "observed_prevalence":
                    observed_prevalence,

                "mean_prediction":
                    mean_prediction,

                # Positive = average overprediction
                "mean_prediction_minus_observed":
                    (
                        mean_prediction
                        -
                        observed_prevalence
                    ),

                "prediction_observed_ratio":
                    (
                        mean_prediction
                        /
                        observed_prevalence
                        if observed_prevalence
                        >
                        0
                        else np.nan
                    ),
            }
        )


        # ----------------------------------------------------
        # Coarse quantile reliability bins
        #
        # duplicates='drop' prevents failure if predictions
        # contain too few distinct values.
        # ----------------------------------------------------

        calibration_df = pd.DataFrame(
            {
                "y":
                    y,

                "p":
                    p,
            }
        )


        calibration_df[
            "bin"
        ] = pd.qcut(
            calibration_df[
                "p"
            ],
            q=
                CALIBRATION_BINS,
            labels=False,
            duplicates="drop",
        )


        bin_summary = (
            calibration_df
            .groupby(
                "bin",
                as_index=False,
            )
            .agg(
                n=(
                    "y",
                    "size",
                ),

                positives=(
                    "y",
                    "sum",
                ),

                mean_prediction=(
                    "p",
                    "mean",
                ),

                observed_frequency=(
                    "y",
                    "mean",
                ),

                min_prediction=(
                    "p",
                    "min",
                ),

                max_prediction=(
                    "p",
                    "max",
                ),
            )
        )


        bin_summary[
            "origin_minutes"
        ] = origin_minutes


        bin_summary[
            "model"
        ] = model_name


        reliability_rows.append(
            bin_summary
        )


calibration_summary = pd.DataFrame(
    calibration_rows
)


reliability_summary = pd.concat(
    reliability_rows,
    ignore_index=True,
)


print(
    "CALIBRATION-IN-THE-LARGE"
)


print(
    "Positive mean_prediction_minus_observed "
    "= average overprediction."
)


display(
    calibration_summary
)


print(
    "\nCOARSE OOF RELIABILITY SUMMARY"
)


display(
    reliability_summary[
        [
            "origin_minutes",
            "model",
            "bin",
            "n",
            "positives",
            "mean_prediction",
            "observed_frequency",
            "min_prediction",
            "max_prediction",
        ]
    ]
)

CALIBRATION-IN-THE-LARGE
Positive mean_prediction_minus_observed = average overprediction.


,origin_minutes,model,n,positives,observed_prevalence,mean_prediction,mean_prediction_minus_observed,prediction_observed_ratio
0,30,LOPO prevalence baseline,232,8,0.034483,0.034206,-0.000276,0.991984
1,30,Direct same-X,232,8,0.034483,0.065908,0.031425,1.911320
2,30,Hierarchical same-X,232,8,0.034483,0.051623,0.017140,1.497074
3,45,LOPO prevalence baseline,254,9,0.035433,0.034036,-0.001397,0.960565
4,45,Direct same-X,254,9,0.035433,0.068807,0.033374,1.941880
5,45,Hierarchical same-X,254,9,0.035433,0.052750,0.017317,1.488725



COARSE OOF RELIABILITY SUMMARY


,origin_minutes,model,bin,n,positives,mean_prediction,observed_frequency,min_prediction,max_prediction
0,30,LOPO prevalence baseline,0,58,5,0.023187,0.086207,0.022099,0.031111
1,30,LOPO prevalence baseline,1,78,3,0.033196,0.038462,0.031746,0.035000
2,30,LOPO prevalence baseline,2,44,0,0.038423,0.000000,0.037037,0.039216
3,30,LOPO prevalence baseline,3,52,0,0.044444,0.000000,0.044444,0.044444
4,30,Direct same-X,0,58,1,0.012121,0.017241,0.001740,0.019435
5,30,Direct same-X,1,58,2,0.026152,0.034483,0.019641,0.033906
6,30,Direct same-X,2,58,3,0.045160,0.051724,0.034159,0.059624
7,30,Direct same-X,3,58,2,0.180198,0.034483,0.059644,0.552117
8,30,Hierarchical same-X,0,58,1,0.004464,0.017241,0.000274,0.009179
9,30,Hierarchical same-X,1,58,1,0.018889,0.017241,0.009258,0.027862


## 4. Corrected temporally aligned results

The corrected experiment compares two formulations for 30-minute future hail occurrence:

$$
\text{Direct:}\qquad
P(H^+ = 1 \mid X^-)
$$

and

$$
\text{Hierarchical:}\qquad
P(S^+ = 1 \mid X^-)
\times
P(H^+ = 1 \mid S^+ = 1, X^-).
$$

Here, \(X^-\) contains only predictors whose valid times do not extend beyond the forecast origin \(t_0\). Pre-\(t_0\) radar predictors use only observations before \(t_0\), while ERA5 fields are retrospective reanalysis variables aligned by valid time rather than operationally available forecast products.

Two pre-specified timing regimes are evaluated:

- **30/30:** 30-minute lookback, forecast origin at +30 min, followed by a 30-minute future target window;
- **45/30:** 30-minute lookback, forecast origin at +45 min, followed by a 30-minute future target window.

The nine-period panel is event-enriched and is therefore treated as a **temporally ordered retrospective development panel**, not as a representative climatological sample or an operational forecasting validation.

### 4.1 Label and sample-frame integrity

After correcting both the MRMS spatial-support calculation and the cross-midnight file-query implementation, all 18 period-origin combinations satisfy the temporal-completeness checks:

- 168 canonical 0.25-degree grid cells per period-origin pair;
- 15 MRMS scans in every pre-origin window;
- 15 MRMS scans in every future target window;
- no duplicate period-origin-grid keys.

Across the 18 future windows, 25 hail-positive grid cells were identified before radar-coverage filtering. Seventeen remained after requiring adequate radar coverage in both the predictor and target windows, while eight hail-positive cells were excluded because radar coverage was insufficient.

Among the 17 radar-coverage-eligible hail-positive cells,

$$
H^+=1,S^+=1: 17,
\qquad
H^+=1,S^+=0: 0.
$$

Thus, within the eligible modeling population, the independently constructed MRMS storm proxy captured all 17 observed hail-positive cells.

Importantly, the storm proxy was constructed **without using NOAA hail labels**. The result above therefore describes hail capture by the independently defined storm rule within the eligible population; it does **not** imply perfect hail capture outside that population.

The final modeling panel contains:

| Regime | Eligible rows | Future storms | Future hail |
|---|---:|---:|---:|
| 30/30 | 232 | 148 | 8 |
| 45/30 | 254 | 149 | 9 |
| **Total** | **486** | **297** | **17** |

The frozen predictor set contains six ERA5 environmental variables and three pre-\(t_0\) radar summaries:

$$
\{
\mathrm{CAPE},
T_{2m},
T_{d,2m},
T_{500},
\mathrm{shear}_{850-500},
\mathrm{shear}_{850-300},
\mathrm{pre\ cmax},
\mathrm{pre\ area5},
\mathrm{pre\ area10}
\}.
$$

No post-\(t_0\) radar quantity is used as a predictor.

### 4.2 Same-\(X\) Direct versus Hierarchical comparison

Both formulations use the same predictor set, the same leave-one-period-out splits, and the same logistic-regression learner.

Pooled held-out results are:

| Regime | Model | PR-AUC | ROC-AUC | Brier | Log loss |
|---|---|---:|---:|---:|---:|
| 30/30 | Direct | 0.0485 | 0.5848 | 0.04431 | 0.18218 |
| 30/30 | Hierarchical | **0.0593** | **0.6602** | **0.03929** | **0.16463** |
| 45/30 | Direct | 0.0519 | 0.5977 | 0.04087 | 0.17334 |
| 45/30 | Hierarchical | **0.0931** | **0.6830** | **0.03525** | **0.15273** |

Using a sign convention in which positive values favor the hierarchy:

| Regime | Δ PR-AUC | Δ ROC-AUC | Δ Brier | Δ Log loss |
|---|---:|---:|---:|---:|
| 30/30 | +0.0107 | +0.0753 | +0.00502 | +0.01755 |
| 45/30 | +0.0412 | +0.0853 | +0.00563 | +0.02062 |

The corrected hierarchy therefore has better pooled PR-AUC, ROC-AUC, Brier score, and log loss than the direct formulation in both timing regimes.

The point estimates of the discrimination gains are larger under 45/30 than under 30/30. This is descriptive rather than evidence of a definitive timing effect: the two regimes have different radar-coverage-eligible row sets, and only 8 and 9 hail-positive cells are available respectively.

### 4.3 Sensitivity to the sampled periods

A paired bootstrap was performed by resampling **whole periods**, rather than individual grid cells, so that within-period grid observations were not treated as independent replicates.

For 30/30, the fraction of valid bootstrap replicates favoring the hierarchy was:

- PR-AUC: 0.818;
- ROC-AUC: 0.878;
- Brier: 0.990;
- Log loss: 0.998.

The central 95% bootstrap interval for the Brier and log-loss improvements remained above zero, whereas the discrimination intervals crossed zero.

For 45/30, the corresponding fractions were:

- PR-AUC: 0.965;
- ROC-AUC: 0.952;
- Brier: 0.975;
- Log loss: 0.920.

For 45/30, the discrimination advantages were positive in most bootstrap resamples, although the lower tails remained close to or slightly below zero.

These bootstrap distributions quantify sensitivity to the composition of the nine sampled periods. They are **not p-values** and are **not confidence intervals for full climatological generalization**.

### 4.4 Where does predictive information enter the hierarchy?

Stage-specific held-out diagnostics show a clear difference between storm prediction and hail-within-storm prediction.

For Stage 1,

$$
P(S^+=1\mid X^-),
$$

the predictor set shows substantial held-out discrimination and positive Brier skill relative to its leakage-free LOPO prevalence baseline:

| Regime | PR-AUC | ROC-AUC | Brier skill vs. LOPO prevalence baseline |
|---|---:|---:|---:|
| 30/30 | 0.768 | 0.767 | +0.245 |
| 45/30 | 0.801 | 0.771 | +0.228 |

For Stage 2,

$$
P(H^+=1\mid S^+=1,X^-),
$$

the predictor set shows some held-out ranking information relative to its no-feature prevalence baseline, but the evidence is much weaker. Stage-2 Brier and log-loss scores are worse than the corresponding training-prevalence baseline in both regimes, and only 8 and 9 positive hail cells are available for evaluation.

A component ablation further clarifies the result.

The Stage-1-only formulation,

$$
P(S^+\mid X^-)
\times
P(H^+\mid S^+=1)_{\text{training prevalence}},
$$

improves probability scores relative to Direct but does not reproduce the full hierarchy's discrimination.

The Stage-2-only formulation,

$$
P(S^+)_{\text{training prevalence}}
\times
P(H^+\mid S^+=1,X^-),
$$

provides a small discrimination improvement at 30/30 but is approximately neutral relative to Direct at 45/30.

The **full hierarchy** produces the strongest PR-AUC and ROC-AUC in both regimes.

This pattern suggests that the full hierarchy's discrimination advantage is not reproduced by either component alone. Stage 1 contributes a physically meaningful storm-occurrence signal, while Stage 2 contributes limited conditional hail ranking; their multiplicative combination produces the strongest held-out discrimination in this development panel.

Given the small number of hail-positive cells, this should be interpreted as evidence of complementary information rather than as a precisely estimated interaction effect.

### 4.5 Probability calibration

The observed hail prevalence is approximately 3.4–3.5% in both timing regimes.

Mean held-out predicted probabilities are:

| Regime | LOPO prevalence baseline | Direct | Hierarchical | Observed |
|---|---:|---:|---:|---:|
| 30/30 | 0.0342 | 0.0659 | 0.0516 | 0.0345 |
| 45/30 | 0.0340 | 0.0688 | 0.0528 | 0.0354 |

The direct formulation therefore overpredicts hail occurrence on average in both regimes.

The hierarchical formulation reduces this mean overprediction substantially:

- at 30/30, mean prediction decreases from approximately 6.6% to 5.2%, compared with an observed prevalence of 3.4%;
- at 45/30, mean prediction decreases from approximately 6.9% to 5.3%, compared with an observed prevalence of 3.5%.

Thus, the hierarchy shows **better calibration-in-the-large than Direct**, in the specific sense that its mean predicted probability is closer to the observed prevalence. This does not establish full calibration across the probability range.

Relative to the leakage-free LOPO prevalence baseline, both learned formulations provide substantially better discrimination.

Probability-score performance is more mixed. The hierarchy is consistently better than Direct, but it does not uniformly outperform the conservative prevalence baseline in Brier score.

This distinction is expected in a rare-event setting: a nearly constant low-probability baseline can obtain a strong Brier score without providing useful row-level discrimination.

Because only 8 and 9 positive outcomes are available, the coarse reliability bins are treated as descriptive diagnostics rather than as a basis for fitting or claiming a stable calibration curve.

### 4.6 Interpretation and scope

The corrected experiment provides **directional evidence** that factorizing future hail occurrence through an independently defined future-storm state can improve held-out prediction relative to a direct hail model using the same temporally ordered predictor set.

The strongest current findings are:

1. the MRMS storm proxy is constructed without using NOAA hail labels and captures all 17 hail-positive cells within the radar-coverage-eligible modeling population;
2. the hierarchical formulation has better pooled PR-AUC, ROC-AUC, Brier score, and log loss than the same-\(X\) direct formulation in both timing regimes;
3. the hierarchy's probability-score advantage over Direct is relatively stable to period-level resampling;
4. future storm occurrence is substantially easier to predict than hail occurrence conditional on storm in this panel;
5. neither stage used alone reproduces the full hierarchy's discrimination advantage;
6. the hierarchy reduces Direct's mean probability overprediction, although its probability forecasts remain imperfect and do not uniformly beat the prevalence baseline on proper scoring rules.

Several limitations prevent population-level claims.

The experiment contains only nine event-enriched periods and 17 eligible hail-positive grid cells. The sample was designed for methodological development rather than to reproduce the natural frequency of ordinary atmospheric conditions.

Eight additional hail-positive grid cells were excluded because radar coverage was inadequate. The reported 17/17 hail capture therefore applies only to the radar-coverage-eligible population.

ERA5 predictors are aligned by valid time but are retrospective reanalysis fields rather than operationally available forecast products.

Leave-one-period-out validation prevents rows from the same period from appearing in both training and testing, but the current panel does not yet provide a large set of independently sampled meteorological events.

Accordingly, these results should be interpreted as a **temporally ordered retrospective development-panel demonstration**, not as final evidence that the hierarchy will outperform a direct model across the full hail climatology or in operational forecasting.

The next population-level step is to evaluate both formulations on the same broad, hail-independent representative panel while preserving the storm-first label construction, pre-\(t_0\) information constraints, and identical-\(X\) comparison.

In [26]:
# ------------------------------------------------------------
# Save final clean Notebook 05 derived outputs
#
# These are derived research artifacts only.
# Raw MRMS / ERA5 / NOAA source data are NOT written here.
# ------------------------------------------------------------

NOTEBOOK05_OUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "tables"
)

NOTEBOOK05_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 1. Final modeling panel
#
# Save only identifiers, timing, frozen predictors, targets,
# and essential eligibility metadata.
# ------------------------------------------------------------

modeling_panel_output_columns = [
    "period_id",
    "panel_group",
    "origin_minutes",
    "period_start_utc",
    "t0",
    "target_end",
    "grid_lat",
    "grid_lon",
    "pre_coverage_fraction",
    "pre_coverage_eligible",
    "future_coverage_fraction",
    "future_coverage_eligible",
    "both_coverage_eligible",
    *FINAL_FEATURES,
    "future_storm",
    "future_hail",
]


modeling_panel_output = (
    final_modeling_panel[
        modeling_panel_output_columns
    ]
    .copy()
)


modeling_panel_output.to_csv(
    NOTEBOOK05_OUT_DIR
    / "temporally_aligned_modeling_panel.csv",
    index=False,
)


# ------------------------------------------------------------
# 2. Held-out OOF predictions
# ------------------------------------------------------------

oof_output = (
    samex_oof_with_baseline[
        [
            "period_id",
            "origin_minutes",
            "grid_lat",
            "grid_lon",
            "future_hail",
            "future_storm",
            "p_climatology",
            "p_direct",
            "p_stage1",
            "p_stage2",
            "p_hierarchical",
        ]
    ]
    .copy()
    .rename(
        columns={
            "p_climatology":
                "p_lopo_prevalence_baseline",
        }
    )
)


oof_output.to_csv(
    NOTEBOOK05_OUT_DIR
    / "temporally_aligned_oof_predictions.csv",
    index=False,
)


# ------------------------------------------------------------
# 3. Primary Direct vs Hierarchical pooled results
# ------------------------------------------------------------

pooled_output = (
    samex_pooled_results
    .copy()
)


pooled_output.to_csv(
    NOTEBOOK05_OUT_DIR
    / "direct_vs_hierarchical_pooled_metrics.csv",
    index=False,
)


paired_output = (
    samex_comparison
    .copy()
)


paired_output.to_csv(
    NOTEBOOK05_OUT_DIR
    / "direct_vs_hierarchical_paired_differences.csv",
    index=False,
)


# ------------------------------------------------------------
# 4. LOPO prevalence-baseline comparison
#
# Rename display labels so the public outputs do not use the
# potentially misleading shorthand "climatology".
# ------------------------------------------------------------

baseline_output = (
    baseline_comparison
    .copy()
)


baseline_output[
    "model"
] = (
    baseline_output[
        "model"
    ]
    .replace(
        {
            "LOPO climatology":
                "LOPO prevalence baseline",
        }
    )
)


baseline_output.to_csv(
    NOTEBOOK05_OUT_DIR
    / "lopo_prevalence_baseline_comparison.csv",
    index=False,
)


baseline_skill_output = (
    skill_vs_climatology
    .copy()
    .rename(
        columns={
            "delta_pr_vs_climatology":
                "delta_pr_vs_prevalence_baseline",

            "delta_roc_vs_climatology":
                "delta_roc_vs_prevalence_baseline",

            "delta_log_loss_vs_climatology":
                "delta_log_loss_vs_prevalence_baseline",
        }
    )
)


baseline_skill_output.to_csv(
    NOTEBOOK05_OUT_DIR
    / "skill_vs_lopo_prevalence_baseline.csv",
    index=False,
)


# ------------------------------------------------------------
# 5. Period-cluster bootstrap summary
# ------------------------------------------------------------

bootstrap_summary.to_csv(
    NOTEBOOK05_OUT_DIR
    / "period_cluster_bootstrap_summary.csv",
    index=False,
)


# ------------------------------------------------------------
# 6. Stage-specific diagnostics
# ------------------------------------------------------------

stage_scores_output = (
    stage_scores
    .copy()
)


stage_scores_output[
    "model"
] = (
    stage_scores_output[
        "model"
    ]
    .replace(
        {
            "Stage-1 LOPO climatology":
                "Stage-1 LOPO prevalence baseline",

            "Stage-2 LOPO climatology":
                "Stage-2 LOPO prevalence baseline",
        }
    )
)


stage_scores_output.to_csv(
    NOTEBOOK05_OUT_DIR
    / "stage_specific_metrics.csv",
    index=False,
)


stage_skill_output = (
    stage_skill
    .copy()
    .rename(
        columns={
            "delta_pr_vs_climatology":
                "delta_pr_vs_prevalence_baseline",

            "delta_roc_vs_climatology":
                "delta_roc_vs_prevalence_baseline",

            "delta_log_loss_vs_climatology":
                "delta_log_loss_vs_prevalence_baseline",
        }
    )
)


stage_skill_output.to_csv(
    NOTEBOOK05_OUT_DIR
    / "stage_specific_skill.csv",
    index=False,
)


# ------------------------------------------------------------
# 7. Hierarchy component ablation
# ------------------------------------------------------------

component_scores.to_csv(
    NOTEBOOK05_OUT_DIR
    / "hierarchy_component_ablation_metrics.csv",
    index=False,
)


component_deltas.to_csv(
    NOTEBOOK05_OUT_DIR
    / "hierarchy_component_ablation_vs_direct.csv",
    index=False,
)


# ------------------------------------------------------------
# 8. Calibration summaries
# ------------------------------------------------------------

calibration_output = (
    calibration_summary
    .copy()
)


calibration_output[
    "model"
] = (
    calibration_output[
        "model"
    ]
    .replace(
        {
            "LOPO climatology":
                "LOPO prevalence baseline",
        }
    )
)


calibration_output.to_csv(
    NOTEBOOK05_OUT_DIR
    / "calibration_in_the_large.csv",
    index=False,
)


reliability_output = (
    reliability_summary
    .copy()
)


reliability_output[
    "model"
] = (
    reliability_output[
        "model"
    ]
    .replace(
        {
            "LOPO climatology":
                "LOPO prevalence baseline",
        }
    )
)


reliability_output.to_csv(
    NOTEBOOK05_OUT_DIR
    / "coarse_reliability_summary.csv",
    index=False,
)


# ------------------------------------------------------------
# 9. Sample-frame / structural audits
# ------------------------------------------------------------

structural_audit.to_csv(
    NOTEBOOK05_OUT_DIR
    / "future_hail_storm_structural_audit.csv",
    index=False,
)


modeling_audit.to_csv(
    NOTEBOOK05_OUT_DIR
    / "modeling_panel_period_summary.csv",
    index=False,
)


lopo_audit.to_csv(
    NOTEBOOK05_OUT_DIR
    / "lopo_fold_feasibility_audit.csv",
    index=False,
)


# ------------------------------------------------------------
# Final output audit
# ------------------------------------------------------------

expected_output_files = [
    "temporally_aligned_modeling_panel.csv",
    "temporally_aligned_oof_predictions.csv",
    "direct_vs_hierarchical_pooled_metrics.csv",
    "direct_vs_hierarchical_paired_differences.csv",
    "lopo_prevalence_baseline_comparison.csv",
    "skill_vs_lopo_prevalence_baseline.csv",
    "period_cluster_bootstrap_summary.csv",
    "stage_specific_metrics.csv",
    "stage_specific_skill.csv",
    "hierarchy_component_ablation_metrics.csv",
    "hierarchy_component_ablation_vs_direct.csv",
    "calibration_in_the_large.csv",
    "coarse_reliability_summary.csv",
    "future_hail_storm_structural_audit.csv",
    "modeling_panel_period_summary.csv",
    "lopo_fold_feasibility_audit.csv",
]


output_audit_rows = []


for filename in (
    expected_output_files
):

    path = (
        NOTEBOOK05_OUT_DIR
        / filename
    )


    output_audit_rows.append(
        {
            "filename":
                filename,

            "exists":
                path.exists(),

            "size_kb":
                (
                    path.stat().st_size
                    / 1024
                    if path.exists()
                    else np.nan
                ),
        }
    )


output_audit = pd.DataFrame(
    output_audit_rows
)


display(
    output_audit
)


print(
    "\nOutput directory:",
    NOTEBOOK05_OUT_DIR,
)


print(
    "Expected files:",
    len(
        expected_output_files
    ),
)


print(
    "Files successfully written:",
    int(
        output_audit[
            "exists"
        ].sum()
    ),
)


print(
    "All expected outputs exist:",
    output_audit[
        "exists"
    ].all(),
)

,filename,exists,size_kb
0,temporally_aligned_modeling_panel.csv,True,113.217773
1,temporally_aligned_oof_predictions.csv,True,67.235352
2,direct_vs_hierarchical_pooled_metrics.csv,True,0.555664
3,direct_vs_hierarchical_paired_differences.csv,True,0.222656
4,lopo_prevalence_baseline_comparison.csv,True,0.820312
5,skill_vs_lopo_prevalence_baseline.csv,True,0.541016
6,period_cluster_bootstrap_summary.csv,True,1.041992
7,stage_specific_metrics.csv,True,1.247070
8,stage_specific_skill.csv,True,0.666992
9,hierarchy_component_ablation_metrics.csv,True,1.330078



Output directory: outputs/tables
Expected files: 16
Files successfully written: 16
All expected outputs exist: True
